# AECD REST API 모델 실험 베이스라인

## Goal

`aecd_platform`을 직접 조회하지 않고 AECD Data API를 통해 비식별 SERS 스펙트럼을 받아, subject 단위로 분할한 기본 모델들을 동일 조건에서 비교합니다.

이 노트북은 연구용 출발점입니다. Cancer Screening 결과는 병원 교란 가능성이 있으며 임상 성능 또는 외부 일반화의 근거로 사용할 수 없습니다.

## Setup

API 실행 예시:

```bash
PGDATABASE=aecd_platform PGPASSWORD='...' uv run --extra api \
  uvicorn sers.aecd_api.app:app --host 127.0.0.1 --port 8000
```

노트북은 기본적으로 `http://127.0.0.1:8000`을 호출합니다. `AECD_API_BASE_URL`로 변경할 수 있습니다. CI나 구조 확인용으로만 `AECD_NOTEBOOK_DEMO=1`을 사용합니다.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from collections import defaultdict

import httpx
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.special import voigt_profile
from scipy.sparse.linalg import spsolve
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 20260819
API_BASE_URL = os.environ.get("AECD_API_BASE_URL", "http://127.0.0.1:8000")
API_KEY = os.environ.get("AECD_API_KEY")
DEMO_MODE = os.environ.get("AECD_NOTEBOOK_DEMO", "0") == "1"
TARGET_COLUMN = "cohort_group"  # "cohort_group" or "cancer_type"
# clinical workbook v7에서 데이터 오너가 Drop으로 지정한 대상은 코호트에서 제외한다.
# subject/sample은 이미 스펙트럼이 붙어 있어 DB에 남고 cohort_group만 Drop으로
# 표시되므로, 거르지 않으면 y_screen = (labels == "prostate") 에서 조용히 비암(0)이
# 된다. 근거: scripts/db/aecd_clinical_v7/README.md (2026-09-02, 오너 확인)
EXCLUDED_COHORT_GROUPS = frozenset({"Drop"})
# Drop 제외 후 보라매 코호트 크기. shape assert가 한 곳만 보고 갱신되도록 상수화.
EXPECTED_SUBJECTS = 112
# CLASS_NAMES 순서(control, prostate disease control, prostate)의 기대 인원.
EXPECTED_CLASS_COUNTS = (20, 49, 43)
SITE_CODE: str | None = None
COHORT_GROUP: str | None = None
CANCER_TYPE: str | None = None
CLINICAL_DB_ENABLED = bool(os.environ.get("PGPASSWORD")) or os.environ.get("AECD_NOTEBOOK_CLINICAL_DB") == "1"
PAGE_SIZE = 500
MAX_SPECTRA = 100000
TEST_SIZE = 0.25
SPECTRAL_RANGE_CM1 = (400.0, 2200.0)
# Keep exact target-grid centers in the registry; do not merge neighboring bins.
PEAK_MATCH_TOLERANCE_CM1 = 0.0
# Working resolution reference; replace with a reference-line FWHM when measured.
INSTRUMENT_RESOLUTION_FWHM_CM1 = 2.0
RESOLUTION_BORDERLINE_FACTOR = 1.25
BASELINE_LAMBDA = 1e5
BASELINE_ASYMMETRY = 0.01
DECONVOLUTION_HALF_WINDOW_CM1 = 20.0
DECONVOLUTION_BIC_IMPROVEMENT = 10.0
DECONVOLUTION_MAX_COMPONENTS = 2

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "stk_v2_preprocess.py").exists() and (
    NOTEBOOK_DIR / "notebooks"
).is_dir():
    NOTEBOOK_DIR /= "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

NOTEBOOK_OUTPUT_DIR = Path(
    os.environ.get(
        "AECD_OUTDIR",
        NOTEBOOK_DIR / "aecd_api_model_baseline_outputs",
    )
)
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print({"api": API_BASE_URL, "demo_mode": DEMO_MODE, "target": TARGET_COLUMN})

{'api': 'http://127.0.0.1:8000', 'demo_mode': False, 'target': 'cohort_group'}


### Key assumptions

- API가 `measurement.raw_spectra`의 acquired clinical measurement만 반환합니다.
- 응답의 `subject_key`는 비식별 내부 키이며 원본 환자 코드는 포함하지 않습니다.
- 동일 subject의 replicate는 평균한 뒤 한 번만 분할하여 replicate leakage를 막습니다.
- 라벨 매핑과 포함 코호트는 위 파라미터에서 명시적으로 결정해야 합니다.

## Steps

### 1. API 연결과 데이터 범위 확인

In [ ]:
def demo_transport() -> httpx.MockTransport:
    rng = np.random.default_rng(RANDOM_STATE)
    grid = np.linspace(400.0, 2200.0, 901)
    labels = ("NOR", "PRO", "BLC")

    # ------------------------------------------------------------
    # Polystyrene(PS) 표준 밴드 = reference peaks
    # (calibrations.reference_peaks_cm1 과 동일한 8개 밴드)
    # ------------------------------------------------------------
    ps_reference_peaks = [
        621.3, 794.8, 1001.2, 1032.0,
        1157.4, 1327.1, 1448.6, 1602.8,
    ]

    reference_peaks = [
        {
            "standard_material": "Polystyrene (PS)",
            "calibration_type": "raman_shift",
            "instrument_name": "DEMO-SERS",
            "reference_peaks_cm1": ps_reference_peaks,
            "tolerance_cm1": 2.0,
            "unit": "cm-1",
        }
    ]

    spectra = []
    measurement_id = 1
    for label_index, label in enumerate(labels):
        for subject_index in range(16):
            center = 500.0 + 90.0 * label_index
            base = np.exp(-0.5 * ((grid - center) / 18.0) ** 2)

            # reference peak 위치에 작은 표준 밴드를 추가
            # (baseline anchor / 정렬 검증용, 신호와 구분 가능하게 약하게)
            ref_signal = np.zeros_like(grid)
            for peak in ps_reference_peaks:
                ref_signal += 0.15 * np.exp(
                    -0.5 * ((grid - peak) / 4.0) ** 2
                )

            for replicate in range(1, 4):
                spectra.append({
                    "measurement_id": measurement_id,
                    "subject_key": f"subject:{label_index * 100 + subject_index}",
                    "sample_key": f"sample:{label_index * 100 + subject_index}",
                    "site_code": "DEMO",
                    "cohort_group": label,
                    "cancer_type": None if label == "NOR" else label.lower(),
                    "measured_at": "2026-08-19T00:00:00Z",
                    "replicate_number": replicate,
                    "instrument_name": "DEMO-SERS",
                    "n_points": len(grid),
                    "x_min": float(grid[0]),
                    "x_max": float(grid[-1]),
                    "reference_peaks_cm1": ps_reference_peaks,
                    "wavenumber": grid.tolist(),
                    "intensities": (
                        base
                        + ref_signal
                        + rng.normal(0.0, 0.03, len(grid))
                    ).tolist(),
                })
                measurement_id += 1

    def handler(request: httpx.Request) -> httpx.Response:
        if request.url.path == "/health":
            return httpx.Response(
                200,
                json={"status": "ok", "database": "aecd_platform"},
            )

        if request.url.path == "/v1/cohorts":
            rows = [
                {
                    "site_code": "DEMO",
                    "cohort_group": label,
                    "cancer_type": None if label == "NOR" else label.lower(),
                    "subjects": 16,
                    "samples": 16,
                    "spectra": 48,
                }
                for label in labels
            ]
            return httpx.Response(200, json=rows)

        # --------------------------------------------------------
        # reference peak 조회 엔드포인트
        # --------------------------------------------------------
        if request.url.path == "/v1/reference_peaks":
            params = request.url.params
            filtered = [
                row for row in reference_peaks
                if (
                    (not params.get("standard_material")
                     or row["standard_material"] == params["standard_material"])
                    and (not params.get("instrument_name")
                         or row["instrument_name"] == params["instrument_name"])
                )
            ]
            return httpx.Response(
                200,
                json={"total": len(filtered), "items": filtered},
            )

        if request.url.path == "/v1/spectra":
            params = request.url.params
            filtered = [row for row in spectra if (
                (not params.get("cohort_group") or row["cohort_group"] == params["cohort_group"])
                and (not params.get("cancer_type") or row["cancer_type"] == params["cancer_type"])
                and (not params.get("site_code") or row["site_code"] == params["site_code"])
            )]
            offset = int(params.get("offset", "0"))
            limit = int(params.get("limit", "100"))
            return httpx.Response(200, json={
                "total": len(filtered),
                "limit": limit,
                "offset": offset,
                "items": filtered[offset:offset + limit],
            })

        return httpx.Response(404)

    return httpx.MockTransport(handler)

In [ ]:
headers = {"X-API-Key": API_KEY} if API_KEY else {}
client_options = {
    "base_url": API_BASE_URL,
    "headers": headers,
    "timeout": httpx.Timeout(30.0, connect=5.0),
    "follow_redirects": True,
}
if DEMO_MODE:
    client_options["transport"] = demo_transport()

with httpx.Client(**client_options) as client:
    health_response = client.get("/health")
    health_response.raise_for_status()

    cohort_response = client.get("/v1/cohorts")
    cohort_response.raise_for_status()
    cohort_summary = pd.DataFrame(cohort_response.json())

    # reference peak 조회
    refpeak_response = client.get(
        "/v1/reference_peaks",
        params={"standard_material": "Polystyrene (PS)"},
    )
    refpeak_response.raise_for_status()
    refpeak_payload = refpeak_response.json()

print(health_response.json())
display(cohort_summary)

# reference peak 배열 추출
REFERENCE_PEAKS_CM1 = np.asarray(
    refpeak_payload["items"][0]["reference_peaks_cm1"],
    dtype=np.float64,
)
TOLERANCE_CM1 = float(
    refpeak_payload["items"][0]["tolerance_cm1"]
)

print("Reference peaks (cm-1):", REFERENCE_PEAKS_CM1)
print("Tolerance (cm-1):", TOLERANCE_CM1)

{'status': 'ok', 'database': 'aecd_platform'}


,site_code,cohort_group,cancer_type,study_cancer_type,diagnosed_cancer_type,case_status,subjects,samples,spectra
0,smcxd07,control,None,None,None,None,21,21,2541
1,smcxd07,prostate,prostate,prostate,prostate,None,43,43,5203
2,smcxd07,prostate disease control,None,None,None,None,49,49,5929


Reference peaks (cm-1): [ 621.3  794.8 1001.2 1032.  1157.4 1327.1 1448.6 1602.8]
Tolerance (cm-1): 2.0


### 2. 페이지 단위 스펙트럼 로딩

In [ ]:
# ============================================================
# API -> AECD notebook format
# ============================================================
def load_spectra_from_api(
    client_options,
    site_code=None,
    cohort_group=None,
    cancer_type=None,
    max_spectra=100000,
    page_size=500,
):
    query_filters = {
        key: value
        for key, value in {
            "site_code": site_code,
            "cohort_group": cohort_group,
            "cancer_type": cancer_type,
        }.items()
        if value is not None
    }

    items = []
    offset = 0

    with httpx.Client(**client_options) as client:

        health = client.get("/health")
        health.raise_for_status()

        print("Health:", health.json())

        while len(items) < max_spectra:

            requested = min(
                page_size,
                max_spectra - len(items)
            )

            response = client.get(
                "/v1/spectra",
                params={
                    **query_filters,
                    "limit": requested,
                    "offset": offset,
                },
            )

            response.raise_for_status()

            page = response.json()

            items.extend(page["items"])

            offset += len(page["items"])

            if (
                not page["items"]
                or offset >= page["total"]
            ):
                break

    if not items:
        raise RuntimeError(
            "API filters returned no spectra"
        )

    excluded = [
        row for row in items
        if row.get("cohort_group") in EXCLUDED_COHORT_GROUPS
    ]
    if excluded:
        items = [
            row for row in items
            if row.get("cohort_group") not in EXCLUDED_COHORT_GROUPS
        ]
        print(
            f"Excluded cohort groups {sorted(EXCLUDED_COHORT_GROUPS)}: "
            f"{len(excluded)} spectra / "
            f"{len({row['subject_key'] for row in excluded})} subjects"
        )
        if not items:
            raise RuntimeError(
                "Every returned spectrum was excluded by cohort group"
            )

    # --------------------------------------------------------
    # metadata
    # --------------------------------------------------------

    metadata = pd.DataFrame([
        {
            key: value
            for key, value in row.items()
            if key not in {
                "wavenumber",
                "intensities",
            }
        }
        for row in items
    ])

    # --------------------------------------------------------
    # target grid
    # --------------------------------------------------------

    target_grid = np.asarray(
        items[0]["wavenumber"],
        dtype=np.float64,
    )

    # --------------------------------------------------------
    # aligned matrix
    # --------------------------------------------------------

    aligned = np.vstack([
        np.asarray(
            row["intensities"],
            dtype=np.float64,
        )
        for row in items
    ])

    # --------------------------------------------------------
    # validation
    # --------------------------------------------------------

    if aligned.shape[1] != len(target_grid):
        raise ValueError(
            "Intensity length and target grid length mismatch"
        )

    # replicate_number -> point_no 호환
    if (
        "point_no" not in metadata.columns
        and "replicate_number" in metadata.columns
    ):
        metadata["point_no"] = (
            metadata["replicate_number"]
        )

    subject_keys = (
        metadata["subject_key"]
        .drop_duplicates()
        .tolist()
    )

    print(
        {
            "loaded_spectra": len(items),
            "api_total": page["total"],
            "subjects": metadata["subject_key"].nunique(),
            "samples": metadata["sample_key"].nunique(),
            "points": aligned.shape[1],
        }
    )

    return (
        aligned,
        metadata,
        subject_keys,
        target_grid,
        items
    )

aligned, metadata, subject_keys, target_grid, items = (
    load_spectra_from_api(
        client_options,
        site_code=None,
        cohort_group=None,
        cancer_type=None,
        max_spectra=50000,
        page_size=1000,
    )
)

print(aligned.shape)
print(metadata.shape)
print(len(subject_keys))
print(target_grid.shape)

Health: {'status': 'ok', 'database': 'aecd_platform'}
{'loaded_spectra': 13673, 'api_total': 13673, 'subjects': 113, 'samples': 113, 'points': 1686}
(13673, 1686)
(13673, 16)
113
(1686,)


### 3. Subject 별 임상정보

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

if CLINICAL_DB_ENABLED:
    db_url = URL.create(
        "postgresql+psycopg2",
        username=os.getenv("PGUSER", "postgres"),
        password=os.getenv("PGPASSWORD", ""),
        host=os.getenv("PGHOST", "localhost"),
        port=int(os.getenv("PGPORT", "5432")),
        database=os.getenv("PGDATABASE", "aecd_platform"),
    )

    engine = create_engine(db_url, pool_pre_ping=True)

    sql = text("""
    SELECT
        current_database() AS database_name,
        inet_server_addr() AS host,
        inet_server_port() AS port,
        (SELECT count(*) FROM master.subjects) AS subjects,
        (SELECT count(*) FROM master.samples) AS samples,
        (SELECT count(*) FROM clinical.diagnoses) AS diagnoses,
        (SELECT count(*) FROM clinical.observations) AS observations
    """)

    display(pd.read_sql_query(sql, engine))
else:
    db_url = None
    engine = None
    print(
        "Clinical DB lookup skipped. Set PGPASSWORD or "
        "AECD_NOTEBOOK_CLINICAL_DB=1 to enable it."
    )


,database_name,host,port,subjects,samples,diagnoses,observations
0,aecd_platform,::1,5432,113,113,113,1016


In [ ]:
if CLINICAL_DB_ENABLED:
    diagnoses_df = pd.read_sql_query(
        text("""
            SELECT
                d.subject_id,
                'subject:' || d.subject_id AS subject_key,
                d.cohort_group,
                d.cancer_type,
                d.diagnosis_name,
                d.pathology_result,
                d.gleason_score,
                d.grade_group,
                d.overall_stage,
                d.tnm_stage_raw
            FROM clinical.diagnoses AS d
            ORDER BY d.subject_id
        """),
        engine,
    )
    
    print(diagnoses_df.shape)
    display(diagnoses_df.head(20))
    display(diagnoses_df["cohort_group"].value_counts(dropna=False))
else:
    diagnoses_df = pd.DataFrame()
    print("diagnoses_df skipped because clinical DB lookup is disabled")


(113, 10)


,subject_id,subject_key,cohort_group,cancer_type,diagnosis_name,pathology_result,gleason_score,grade_group,overall_stage,tnm_stage_raw
0,1,subject:1,control,None,None,None,None,NaN,None,None
1,2,subject:2,prostate disease control,None,None,None,None,NaN,None,None
2,3,subject:3,prostate disease control,None,None,None,None,NaN,None,None
3,4,subject:4,control,None,None,None,None,NaN,None,None
4,5,subject:5,prostate disease control,None,None,None,None,NaN,None,None
5,6,subject:6,control,None,None,None,None,NaN,None,None
6,7,subject:7,prostate disease control,None,None,None,None,NaN,None,None
7,8,subject:8,prostate disease control,None,None,None,None,NaN,None,None
8,9,subject:9,prostate disease control,None,None,None,None,NaN,None,None
9,10,subject:10,prostate disease control,None,None,None,None,NaN,None,None


cohort_group
prostate disease control    49
prostate                    43
control                     21
Name: count, dtype: int64

In [ ]:
if CLINICAL_DB_ENABLED:
    observations_df = pd.read_sql_query(
        text("""
            SELECT
                o.subject_id,
                'subject:' || o.subject_id AS subject_key,
                o.sample_id,
                'sample:' || o.sample_id AS sample_key,
                o.panel,
                o.code,
                o.raw_value,
                o.numeric_value,
                o.unit,
                o.normalization_status
            FROM clinical.observations AS o
            WHERE lower(o.code) IN (
                'psa',
                'ua_ph',
                'ua_sg',
                'ua_protein',
                'microscopy_wbc',
                'microscopy_rbc'
            )
            ORDER BY o.subject_id, o.code
        """),
        engine,
    )
    
    print(observations_df.shape)
    display(observations_df.head(30))
    display(observations_df["code"].value_counts())
else:
    observations_df = pd.DataFrame()
    print("observations_df skipped because clinical DB lookup is disabled")


(678, 10)


,subject_id,subject_key,sample_id,sample_key,panel,code,raw_value,numeric_value,unit,normalization_status
0,1,subject:1,1,sample:1,urine_microscopy,Microscopy_RBC,1-4,None,None,raw_only
1,1,subject:1,1,sample:1,urine_microscopy,Microscopy_WBC,1-4,None,None,raw_only
2,1,subject:1,1,sample:1,tumor_marker,psa,1.17,None,None,raw_only
3,1,subject:1,1,sample:1,urinalysis,ua_ph,5.5,None,None,raw_only
4,1,subject:1,1,sample:1,urinalysis,ua_protein,Trace,None,None,raw_only
5,1,subject:1,1,sample:1,urinalysis,ua_sg,1.031,None,None,raw_only
6,2,subject:2,2,sample:2,urine_microscopy,Microscopy_RBC,<1,None,None,raw_only
7,2,subject:2,2,sample:2,urine_microscopy,Microscopy_WBC,<1,None,None,raw_only
8,2,subject:2,2,sample:2,tumor_marker,psa,8.64,None,None,raw_only
9,2,subject:2,2,sample:2,urinalysis,ua_ph,5.5,None,None,raw_only


code
Microscopy_RBC    113
Microscopy_WBC    113
psa               113
ua_ph             113
ua_protein        113
ua_sg             113
Name: count, dtype: int64

In [ ]:
if CLINICAL_DB_ENABLED:
    clinical_df = pd.read_sql_query(
        text("""
            SELECT
                sub.subject_id,
                'subject:' || sub.subject_id AS subject_key,
                sample.sample_id,
                'sample:' || sample.sample_id AS sample_key,
                sample.solum_label,
                site.site_code,
                sub.sex,
                sample.collection_date,
    
                d.cohort_group,
                d.cancer_type,
                d.gleason_score,
                d.grade_group,
                d.overall_stage,
    
                MAX(o.raw_value)
                    FILTER (WHERE lower(o.code) = 'psa') AS psa_raw,
    
                MAX(o.numeric_value)
                    FILTER (WHERE lower(o.code) = 'psa') AS psa,
    
                MAX(o.raw_value)
                    FILTER (WHERE lower(o.code) = 'ua_ph') AS ua_ph,
    
                MAX(o.raw_value)
                    FILTER (WHERE lower(o.code) = 'ua_sg') AS ua_sg,
    
                MAX(o.raw_value)
                    FILTER (WHERE lower(o.code) = 'microscopy_wbc')
                    AS microscopy_wbc,
    
                MAX(o.raw_value)
                    FILTER (WHERE lower(o.code) = 'microscopy_rbc')
                    AS microscopy_rbc
    
            FROM master.subjects AS sub
            JOIN master.sites AS site
              ON site.site_id = sub.site_id
            JOIN master.samples AS sample
              ON sample.subject_id = sub.subject_id
            LEFT JOIN clinical.diagnoses AS d
              ON d.subject_id = sub.subject_id
            LEFT JOIN clinical.observations AS o
              ON o.subject_id = sub.subject_id
             AND (
                  o.sample_id = sample.sample_id
                  OR o.sample_id IS NULL
             )
    
            WHERE site.site_code = 'smcxd07'
    
            GROUP BY
                sub.subject_id,
                sample.sample_id,
                sample.solum_label,
                site.site_code,
                sub.sex,
                sample.collection_date,
                d.cohort_group,
                d.cancer_type,
                d.gleason_score,
                d.grade_group,
                d.overall_stage
    
            ORDER BY sample.solum_label
        """),
        engine,
    )
    
    print("clinical_df:", clinical_df.shape)
    display(clinical_df.head(20))
    display(clinical_df["cohort_group"].value_counts(dropna=False))
    clinical_df.to_excel(NOTEBOOK_OUTPUT_DIR / "clinical_df.xlsx", index=False)
else:
    clinical_df = pd.DataFrame()
    print("clinical_df skipped because clinical DB lookup is disabled")


clinical_df: (113, 19)


,subject_id,subject_key,sample_id,sample_key,solum_label,site_code,sex,collection_date,cohort_group,cancer_type,gleason_score,grade_group,overall_stage,psa_raw,psa,ua_ph,ua_sg,microscopy_wbc,microscopy_rbc
0,1,subject:1,1,sample:1,BNOR_1,smcxd07,남성,2026-03-03,control,None,None,NaN,None,1.17,None,5.5,1.031,1-4,1-4
1,51,subject:51,51,sample:51,BNOR_100,smcxd07,남성,2026-06-04,control,None,None,NaN,None,5.29,None,6,1.016,1-4,<1
2,52,subject:52,52,sample:52,BNOR_102,smcxd07,남성,2026-06-04,prostate disease control,None,None,NaN,None,4.07,None,7.5,1.02,<1,<1
3,53,subject:53,53,sample:53,BNOR_103,smcxd07,남성,2026-06-08,prostate disease control,None,None,NaN,None,10.2,None,5,1.023,>=100,10-19
4,54,subject:54,54,sample:54,BNOR_106,smcxd07,남성,2026-06-08,prostate disease control,None,None,NaN,None,3.9,None,6,1.006,<1,<1
5,55,subject:55,55,sample:55,BNOR_110,smcxd07,남성,2026-06-15,control,None,None,NaN,None,3.91,None,7,1.012,<1,5-9
6,56,subject:56,56,sample:56,BNOR_114,smcxd07,남성,2026-06-18,prostate disease control,None,None,NaN,None,4.8,None,5,1.016,<1,<1
7,57,subject:57,57,sample:57,BNOR_116,smcxd07,남성,2026-06-18,prostate disease control,None,None,NaN,None,6.19,None,5.5,1.025,<1,<1
8,58,subject:58,58,sample:58,BNOR_119,smcxd07,남성,2026-06-19,prostate disease control,None,None,NaN,None,9.9,None,6.5,1.021,<1,<1
9,6,subject:6,6,sample:6,BNOR_14,smcxd07,남성,2026-03-16,control,None,None,NaN,None,0.99,None,5.5,1.038,<1,<1


cohort_group
prostate disease control    49
prostate                    43
control                     21
Name: count, dtype: int64

### 3. 공통 파수 격자 정렬과 subject 평균

In [ ]:
source_grids = [np.asarray(row["wavenumber"], dtype=np.float64) for row in items]
source_values = [np.asarray(row["intensities"], dtype=np.float64) for row in items]
spectral_min, spectral_max = SPECTRAL_RANGE_CM1
coverage_failures = [
    index
    for index, grid in enumerate(source_grids)
    if grid.ndim != 1 or len(grid) < 2 or grid[0] > spectral_min or grid[-1] < spectral_max
]
if coverage_failures:
    raise RuntimeError(
        f"{len(coverage_failures)} spectra do not cover {spectral_min:.0f}-{spectral_max:.0f} cm^-1"
    )
if any(len(grid) != len(values) for grid, values in zip(source_grids, source_values, strict=True)):
    raise RuntimeError("Wavenumber and intensity lengths differ")
if any(np.any(np.diff(grid) <= 0) for grid in source_grids):
    raise RuntimeError("Wavenumber grids must be strictly increasing")

points_in_range = [
    int(np.count_nonzero((grid >= spectral_min) & (grid <= spectral_max)))
    for grid in source_grids
]
common_points = min(points_in_range)
if common_points < 2:
    raise RuntimeError("Spectra do not share a usable 400-2200 cm^-1 range")
target_grid = np.linspace(spectral_min, spectral_max, common_points)
aligned = np.vstack([
    np.interp(target_grid, grid, values)
    for grid, values in zip(source_grids, source_values, strict=True)
])

subject_rows: dict[str, list[int]] = defaultdict(list)
for row_index, subject_key in enumerate(metadata["subject_key"]):
    subject_rows[subject_key].append(row_index)

subject_spectra = []
subject_labels = []
subject_keys = []
for subject_key, row_indices in subject_rows.items():
    labels = metadata.iloc[row_indices][TARGET_COLUMN].dropna().unique()
    if len(labels) != 1:
        continue
    subject_keys.append(subject_key)
    subject_labels.append(str(labels[0]))
    subject_spectra.append(aligned[row_indices].mean(axis=0))

X_RAW_SUBJECT_MEAN = np.vstack(subject_spectra)
# X is replaced with baseline-corrected + area-normalized data before peak work.
X = X_RAW_SUBJECT_MEAN.copy()
y = np.asarray(subject_labels)
print({
    "X_shape": X.shape,
    "range_cm-1": (float(target_grid[0]), float(target_grid[-1])),
    "grid_step_cm-1": float(np.median(np.diff(target_grid))),
    "classes": dict(zip(*np.unique(y, return_counts=True), strict=True)),
})

{'X_shape': (113, 933), 'range_cm-1': (400.0, 2200.0), 'grid_step_cm-1': 1.9313304721030136, 'classes': {np.str_('control'): np.int64(21), np.str_('prostate'): np.int64(43), np.str_('prostate disease control'): np.int64(49)}}


### 4. Raw peak 후보와 실제 Voigt deconvolution

`peak_registry`는 smoothing 없이 `X`에서 검출한 후보 위치입니다. 최종 peak 개수는 registry 행 개수가 아니라, 각 실제 spectrum의 region에서 1개/2개 Voigt 모델을 BIC로 비교한 결과로 판단합니다. `X_RAW_SUBJECT_MEAN`은 replicate를 subject별 평균한 raw/aligned spectrum이고, peak 분석용 `X`에는 AsLS baseline correction과 area normalization만 적용합니다. Savitzky-Golay/convolution smoothing은 사용하지 않습니다.

### STK-V2 preprocessing module

STK-V2 전처리는 별도 모듈 [`stk_v2_preprocess.py`](stk_v2_preprocess.py)에서 관리합니다. 이 노트북은 해당 모듈의 공개 함수만 호출해 모델 입력과 peak feature를 생성합니다.


In [ ]:
import numpy as np
from stk_v2_preprocess import stk_v2_channels, stk_v2_peak_input

# 933-point 원본 x축 복원
source_grid = np.linspace(400.0, 2200.0, X_RAW_SUBJECT_MEAN.shape[1])  # (933,)

# 각 subject 원본에서 3채널(ch0/d1/d2) 생성 → (EXPECTED_SUBJECTS, 3, 935)
channels = np.stack([
    stk_v2_channels(source_grid, row)   # 시그니처: (wavenumber, intensity)
    for row in X_RAW_SUBJECT_MEAN
])

X    = channels[:, 0, :]   # ch0  (raw view) - 기존 X와 동일
X_d1 = channels[:, 1, :]   # 1차 미분 + SNV
X_d2 = channels[:, 2, :]   # 2차 미분 + SNV
print("X", X.shape, "| X_d1", X_d1.shape, "| X_d2", X_d2.shape)

X (113, 935) | X_d1 (113, 935) | X_d2 (113, 935)


In [ ]:
# subject_cohort_groups: X_RAW_SUBJECT_MEAN 행 순서와 동일한 cohort_group 리스트/배열 (길이 EXPECTED_SUBJECTS)
# 예: ['control', 'prostate', 'prostate disease control', ...]
subject_cohort_groups = metadata.groupby("subject_key")[TARGET_COLUMN].first().reindex(metadata["subject_key"].unique()).tolist()
labels_raw = np.asarray(subject_cohort_groups)

y_screen = (labels_raw == "prostate").astype(int)   # 1=cancer, 0=non-cancer
assert len(y_screen) == len(X) == EXPECTED_SUBJECTS
print("cancer:", int(y_screen.sum()), "/ non-cancer:", int((1 - y_screen).sum()))
# 기대: cancer 43 / non-cancer 69

cancer: 43 / non-cancer: 70


In [ ]:
import numpy as np

from stk_v2_preprocess import (
    STK_TARGET_GRID,
    stk_v2_channels,
)
from build_stkv2_dataset import extract_peak_features


# ============================================================
# 1. 입력 파수축 결정
# ============================================================

X_raw = np.asarray(X_RAW_SUBJECT_MEAN, dtype=float)

if X_raw.shape[1] == 933:
    # 기존 aligned가 사용했던 실제 933-point 축
    source_grid = np.linspace(400.0, 2200.0, 933)

elif X_raw.shape[1] == 935:
    # 이미 STK-V2 grid로 변환된 경우
    source_grid = STK_TARGET_GRID.copy()

else:
    raise ValueError(
        f"지원하지 않는 spectrum 길이: {X_raw.shape[1]}"
    )

assert len(source_grid) == X_raw.shape[1]


# ============================================================
# 2. STK-V2 3채널 생성
# ============================================================

channels = np.stack([
    stk_v2_channels(source_grid, row)
    for row in X_raw
])

X = channels[:, 0, :]       # ch0
X_d1 = channels[:, 1, :]    # 1차 미분 + SNV
X_d2 = channels[:, 2, :]    # 2차 미분 + SNV

print("channels:", channels.shape)
print("X:", X.shape)
print("X_d1:", X_d1.shape)
print("X_d2:", X_d2.shape)

assert channels.shape == (EXPECTED_SUBJECTS, 3, 935)


# ============================================================
# 3. Peak feature 생성
# ============================================================

X_peak = np.vstack([
    extract_peak_features(STK_TARGET_GRID, row)
    for row in X
])

print("X_peak:", X_peak.shape)

assert X_peak.shape == (EXPECTED_SUBJECTS, 75)
assert np.isfinite(X_peak).all()


# ============================================================
# 4. Subject label 순서 정렬
# ============================================================

subject_order = np.asarray(subject_keys)

label_by_subject = (
    metadata
    .groupby("subject_key", sort=False)[TARGET_COLUMN]
    .first()
)

labels_raw = (
    label_by_subject
    .reindex(subject_order)
    .to_numpy(dtype=str)
)

assert len(labels_raw) == len(X) == len(X_peak)


# ============================================================
# 5. Binary label: Cancer vs Non-cancer
# ============================================================

y_screen = (labels_raw == "prostate").astype(int)

print(
    "Binary:",
    f"cancer = {int(y_screen.sum())}",
    f"| non-cancer = {int((y_screen == 0).sum())}",
)


# ============================================================
# 6. Three-class label
# 0 = Control
# 1 = Elevated PSA / Bx-
# 2 = Prostate Cancer
# ============================================================

CLASS_MAP = {
    "control": 0,
    "prostate disease control": 1,
    "prostate": 2,
}

CLASS_NAMES = np.array([
    "Control",
    "Elevated PSA/Bx-",
    "Prostate Cancer",
])

unknown_labels = sorted(set(labels_raw) - set(CLASS_MAP))

if unknown_labels:
    raise ValueError(f"정의되지 않은 label: {unknown_labels}")

y_3class = np.array(
    [CLASS_MAP[label] for label in labels_raw],
    dtype=int,
)

for class_index, class_name in enumerate(CLASS_NAMES):
    print(
        class_index,
        class_name,
        "=",
        int(np.sum(y_3class == class_index)),
    )

assert np.array_equal(
    np.bincount(y_3class, minlength=3),
    np.array(EXPECTED_CLASS_COUNTS),
)

channels: (113, 3, 935)
X: (113, 935)
X_d1: (113, 935)
X_d2: (113, 935)
X_peak: (113, 75)
Binary: cancer = 43 | non-cancer = 70
0 Control = 21
1 Elevated PSA/Bx- = 49
2 Prostate Cancer = 43


### 분석 성능 확인

In [ ]:
from train_stkv2_stacking import (
    build_views,
    run_nested_cv,
    DECISION_THRESHOLD,
)

print("threshold:", DECISION_THRESHOLD)


threshold: 0.4


In [ ]:
data_stkv2 = {
    "raw": X,
    "d1": X_d1,
    "d2": X_d2,
    "peak": X_peak,
}

views = build_views(data_stkv2)
groups = np.asarray(subject_order)

for name, matrix in views.items():
    print(name, matrix.shape)

assert views["raw"].shape == (EXPECTED_SUBJECTS, 935)
assert views["d1"].shape == (EXPECTED_SUBJECTS, 935)
assert views["d2"].shape == (EXPECTED_SUBJECTS, 935)
assert views["concat"].shape == (EXPECTED_SUBJECTS, 2805)
assert views["peak"].shape == (EXPECTED_SUBJECTS, 75)

assert len(y_screen) == len(groups) == EXPECTED_SUBJECTS
assert len(np.unique(groups)) == EXPECTED_SUBJECTS

oof_screen, fold_metrics_screen = run_nested_cv(
    views=views,
    y=y_screen,
    groups=groups,
    n_outer=5,
    n_inner=5,
    seed=0,
)

raw (113, 935)
d1 (113, 935)
d2 (113, 935)
concat (113, 2805)
peak (113, 75)


c:\Users\user\anaconda3\envs\sers-analysis\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


  [outer 1] AUC=0.8500  balanced_acc=0.7125


c:\Users\user\anaconda3\envs\sers-analysis\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


  [outer 2] AUC=0.5583  balanced_acc=0.5833


...

  [outer 3] AUC=0.8571  balanced_acc=0.7063


c:\Users\user\anaconda3\envs\sers-analysis\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


  [outer 4] AUC=0.6838  balanced_acc=0.5299


c:\Users\user\anaconda3\envs\sers-analysis\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


  [outer 5] AUC=0.8120  balanced_acc=0.7735


In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)

fold_metrics_screen = np.asarray(fold_metrics_screen)

fold_auc = fold_metrics_screen[:, 0]
fold_bacc = fold_metrics_screen[:, 1]

pred_screen = (
    oof_screen >= DECISION_THRESHOLD
).astype(int)

print("===== STK-V2 Cancer Screening =====")
print(f"Fold AUC      : {fold_auc.mean():.4f} ± {fold_auc.std():.4f}")
print(f"Fold BAcc     : {fold_bacc.mean():.4f} ± {fold_bacc.std():.4f}")
print(f"Overall AUC   : {roc_auc_score(y_screen, oof_screen):.4f}")
print(
    "Overall BAcc :",
    f"{balanced_accuracy_score(y_screen, pred_screen):.4f}",
)

print("\nConfusion matrix:")
print(confusion_matrix(y_screen, pred_screen))

print("\nClassification report:")
print(
    classification_report(
        y_screen,
        pred_screen,
        target_names=["Non-cancer", "Cancer"],
        digits=4,
    )
)

===== STK-V2 Cancer Screening =====
Fold AUC      : 0.7522 ± 0.1153
Fold BAcc     : 0.6611 ± 0.0901
Overall AUC   : 0.7405
Overall BAcc : 0.6621

Confusion matrix:
[[52 18]
 [18 25]]

Classification report:
              precision    recall  f1-score   support

  Non-cancer     0.7429    0.7429    0.7429        70
      Cancer     0.5814    0.5814    0.5814        43

    accuracy                         0.6814       113
   macro avg     0.6621    0.6621    0.6621       113
weighted avg     0.6814    0.6814    0.6814       113



### STK-V2 binary ROC and fold stability

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

overall_screen_auc = float(roc_auc_score(y_screen, oof_screen))
fpr_screen, tpr_screen, _ = roc_curve(y_screen, oof_screen)
fold_ids = np.arange(1, len(fold_auc) + 1)

fig, (ax_roc, ax_fold) = plt.subplots(1, 2, figsize=(10, 4.4))

ax_roc.plot(
    fpr_screen,
    tpr_screen,
    color="#1f77b4",
    linewidth=2,
    label=f"Overall subject-level OOF AUC = {overall_screen_auc:.3f}",
)
ax_roc.plot([0, 1], [0, 1], "k--", linewidth=1)
ax_roc.set_title("Overall OOF ROC")
ax_roc.set_xlabel("1 - Specificity")
ax_roc.set_ylabel("Sensitivity")
ax_roc.set_xlim(0, 1)
ax_roc.set_ylim(0, 1)
ax_roc.grid(alpha=0.25)
ax_roc.legend(loc="lower right", fontsize=8)

ax_fold.axhline(0.5, color="#777777", linestyle="--", linewidth=1)
ax_fold.axhline(
    float(fold_auc.mean()),
    color="#d62728",
    linestyle=":",
    linewidth=1.5,
    label=f"Mean = {fold_auc.mean():.3f}",
)
ax_fold.plot(fold_ids, fold_auc, "o-", color="#d62728", linewidth=1.8)
for fold_id, auc in zip(fold_ids, fold_auc):
    ax_fold.annotate(
        f"{auc:.3f}",
        (fold_id, auc),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        fontsize=8,
    )
ax_fold.set_title("Outer-fold AUC")
ax_fold.set_xlabel("Outer fold")
ax_fold.set_ylabel("ROC-AUC")
ax_fold.set_xticks(fold_ids)
ax_fold.set_ylim(0.45, 1.0)
ax_fold.grid(alpha=0.25, axis="y")
ax_fold.legend(loc="lower right", fontsize=8)

fig.suptitle("STK-V2 Cancer Screening: OOF ROC and fold AUC")
fig.text(
    0.5,
    0.01,
    "Research-use only. Cancer Screening AUC may be hospital-confounded.",
    ha="center",
    fontsize=8,
    color="#555555",
)
fig.tight_layout(rect=[0, 0.08, 1, 0.94])
figure_path = Path(NOTEBOOK_OUTPUT_DIR) / "E_stk_v2_screening_roc_fold_auc.png"
fig.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved: {figure_path}")

In [ ]:
import pandas as pd

screening_results = pd.DataFrame({
    "subject_key": subject_order,
    "clinical_group": labels_raw,
    "true_cancer": y_screen,
    "cancer_probability": oof_screen,
    "predicted_cancer": pred_screen,
})

screening_results["correct"] = (
    screening_results["true_cancer"]
    == screening_results["predicted_cancer"]
)

display(screening_results.head())
display(
    pd.crosstab(
        screening_results["clinical_group"],
        screening_results["predicted_cancer"],
        margins=True,
    )
)

,subject_key,clinical_group,true_cancer,cancer_probability,predicted_cancer,correct
0,subject:1,control,0,0.212573,0,True
1,subject:2,prostate disease control,0,0.212072,0,True
2,subject:3,prostate disease control,0,0.160567,0,True
3,subject:4,control,0,0.217990,0,True
4,subject:5,prostate disease control,0,0.187711,0,True


predicted_cancer,0,1,All
clinical_group,,,
control,13,8,21
prostate,18,25,43
prostate disease control,39,10,49
All,70,43,113


In [ ]:
display(
    screening_results
    .groupby("clinical_group")["cancer_probability"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

,count,mean,median,std,min,max
clinical_group,,,,,,
control,21,0.347869,0.279148,0.172086,0.166182,0.692899
prostate,43,0.474760,0.473482,0.188686,0.158407,0.789237
prostate disease control,49,0.315543,0.243110,0.160801,0.158768,0.755528


### 3군 임상 성능 확인

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score,
)

from xgboost import XGBClassifier

In [ ]:
N_CLASSES = 3
RANDOM_STATE = 0


def make_multiclass_base_models(seed=0):

    def lr():
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=3000,
                C=1.0,
                solver="lbfgs",
                random_state=seed,
            ),
        )

    def ridge():
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=3000,
                C=0.1,
                solver="lbfgs",
                random_state=seed,
            ),
        )

    def xgb():
        return XGBClassifier(
            objective="multi:softprob",
            num_class=N_CLASSES,
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="mlogloss",
            n_jobs=-1,
            random_state=seed,
            verbosity=0,
        )

    def rf():
        return RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            n_jobs=-1,
            random_state=seed,
        )

    return [
        ("lr_raw",       "raw",    lr),
        ("lr_d1",        "d1",     lr),
        ("lr_d2",        "d2",     lr),
        ("lr_concat",    "concat", lr),
        ("lr_peak",      "peak",   lr),
        ("xgb_raw",      "raw",    xgb),
        ("xgb_d1",       "d1",     xgb),
        ("rf_raw",       "raw",    rf),
        ("rf_d1",        "d1",     rf),
        ("ridge_concat", "concat", ridge),
    ]

In [ ]:
def aligned_predict_proba(model, X, classes=np.arange(N_CLASSES)):
    probability = model.predict_proba(X)
    aligned = np.zeros((len(X), len(classes)), dtype=float)

    model_classes = model.classes_

    for source_column, class_label in enumerate(model_classes):
        target_column = np.where(classes == class_label)[0][0]
        aligned[:, target_column] = probability[:, source_column]

    return aligned

In [ ]:
def run_nested_cv_multiclass(
    views,
    y,
    groups,
    n_outer=5,
    n_inner=5,
    seed=0,
):
    y = np.asarray(y, dtype=int)
    groups = np.asarray(groups)

    classes = np.arange(N_CLASSES)
    base_specs = make_multiclass_base_models(seed)

    n_subjects = len(y)
    n_models = len(base_specs)
    n_meta_features = n_models * N_CLASSES

    oof_probability = np.full(
        (n_subjects, N_CLASSES),
        np.nan,
        dtype=float,
    )

    fold_metrics = []

    outer_cv = StratifiedGroupKFold(
        n_splits=n_outer,
        shuffle=True,
        random_state=seed,
    )

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(views["raw"], y, groups),
        start=1,
    ):
        y_train = y[train_idx]
        groups_train = groups[train_idx]

        inner_cv = StratifiedGroupKFold(
            n_splits=n_inner,
            shuffle=True,
            random_state=seed + fold,
        )

        # 10 models × 3 class probabilities = 30 meta features
        meta_train = np.zeros(
            (len(train_idx), n_meta_features),
            dtype=float,
        )

        for model_index, (name, view_key, factory) in enumerate(base_specs):
            X_view = views[view_key]
            model_oof = np.zeros(
                (len(train_idx), N_CLASSES),
                dtype=float,
            )

            for inner_train, inner_valid in inner_cv.split(
                X_view[train_idx],
                y_train,
                groups_train,
            ):
                model = factory()

                model.fit(
                    X_view[train_idx][inner_train],
                    y_train[inner_train],
                )

                model_oof[inner_valid] = aligned_predict_proba(
                    model,
                    X_view[train_idx][inner_valid],
                    classes,
                )

            start = model_index * N_CLASSES
            end = start + N_CLASSES
            meta_train[:, start:end] = model_oof

        # Multiclass ElasticNet meta learner
        meta_model = LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=0.5,
            C=1.0,
            max_iter=10000,
            random_state=seed + fold,
        )

        meta_model.fit(meta_train, y_train)

        # Outer test용 base probability
        meta_test = np.zeros(
            (len(test_idx), n_meta_features),
            dtype=float,
        )

        for model_index, (name, view_key, factory) in enumerate(base_specs):
            X_view = views[view_key]
            model = factory()

            model.fit(
                X_view[train_idx],
                y_train,
            )

            probability = aligned_predict_proba(
                model,
                X_view[test_idx],
                classes,
            )

            start = model_index * N_CLASSES
            end = start + N_CLASSES
            meta_test[:, start:end] = probability

        test_probability = aligned_predict_proba(
            meta_model,
            meta_test,
            classes,
        )

        oof_probability[test_idx] = test_probability

        test_prediction = np.argmax(
            test_probability,
            axis=1,
        )

        fold_auc = roc_auc_score(
            y[test_idx],
            test_probability,
            multi_class="ovr",
            average="macro",
            labels=classes,
        )

        fold_bacc = balanced_accuracy_score(
            y[test_idx],
            test_prediction,
        )

        fold_f1 = f1_score(
            y[test_idx],
            test_prediction,
            average="macro",
        )

        fold_metrics.append({
            "fold": fold,
            "macro_auc_ovr": fold_auc,
            "balanced_accuracy": fold_bacc,
            "macro_f1": fold_f1,
        })

        print(
            f"[outer {fold}] "
            f"macro AUC={fold_auc:.4f} | "
            f"BAcc={fold_bacc:.4f} | "
            f"macro F1={fold_f1:.4f}"
        )

    if np.isnan(oof_probability).any():
        raise RuntimeError("OOF probability에 미할당 값이 있습니다.")

    return oof_probability, pd.DataFrame(fold_metrics)

In [ ]:
data_stkv2 = {
    "raw": X,
    "d1": X_d1,
    "d2": X_d2,
    "peak": X_peak,
}

views = {
    "raw": data_stkv2["raw"],
    "d1": data_stkv2["d1"],
    "d2": data_stkv2["d2"],
    "concat": np.concatenate(
        [
            data_stkv2["raw"],
            data_stkv2["d1"],
            data_stkv2["d2"],
        ],
        axis=1,
    ),
    "peak": data_stkv2["peak"],
}

groups = np.asarray(subject_order)

for name, matrix in views.items():
    print(name, matrix.shape)

print("y_3class:", y_3class.shape)
print("groups:", groups.shape)
print("class counts:", np.bincount(y_3class))

assert views["raw"].shape == (EXPECTED_SUBJECTS, 935)
assert views["concat"].shape == (EXPECTED_SUBJECTS, 2805)
assert views["peak"].shape == (EXPECTED_SUBJECTS, 75)
assert np.array_equal(
    np.bincount(y_3class),
    np.array(EXPECTED_CLASS_COUNTS),
)

raw (113, 935)
d1 (113, 935)
d2 (113, 935)
concat (113, 2805)
peak (113, 75)
y_3class: (113,)
groups: (113,)
class counts: [21 49 43]


In [ ]:
oof_3class, fold_metrics_3class = run_nested_cv_multiclass(
    views=views,
    y=y_3class,
    groups=groups,
    n_outer=5,
    n_inner=5,
    seed=0,
)
pred_3class = np.argmax(oof_3class, axis=1)

overall_macro_auc = roc_auc_score(
    y_3class,
    oof_3class,
    multi_class="ovr",
    average="macro",
    labels=np.arange(N_CLASSES),
)

overall_weighted_auc = roc_auc_score(
    y_3class,
    oof_3class,
    multi_class="ovr",
    average="weighted",
    labels=np.arange(N_CLASSES),
)

overall_bacc = balanced_accuracy_score(
    y_3class,
    pred_3class,
)

overall_macro_f1 = f1_score(
    y_3class,
    pred_3class,
    average="macro",
)

print("===== STK-V2 Three-Class 결과 =====")
print(f"Macro OvR AUC : {overall_macro_auc:.4f}")
print(f"Weighted AUC  : {overall_weighted_auc:.4f}")
print(f"Balanced Acc  : {overall_bacc:.4f}")
print(f"Macro F1      : {overall_macro_f1:.4f}")

display(fold_metrics_3class)

print("\nConfusion matrix:")
print(confusion_matrix(y_3class, pred_3class))

print("\nClassification report:")
print(
    classification_report(
        y_3class,
        pred_3class,
        target_names=CLASS_NAMES,
        digits=4,
    )
)

c:\Users\user\anaconda3\envs\sers-analysis\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[outer 1] macro AUC=0.5926 | BAcc=0.4444 | macro F1=0.3923


c:\Users\user\anaconda3\envs\sers-analysis\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[outer 2] macro AUC=0.6501 | BAcc=0.4259 | macro F1=0.3783


c:\Users\user\anaconda3\envs\sers-analysis\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


...

[outer 4] macro AUC=0.5341 | BAcc=0.3750 | macro F1=0.3421


c:\Users\user\anaconda3\envs\sers-analysis\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[outer 5] macro AUC=0.6306 | BAcc=0.3500 | macro F1=0.3001
===== STK-V2 Three-Class 결과 =====
Macro OvR AUC : 0.6260
Weighted AUC  : 0.6344
Balanced Acc  : 0.4183
Macro F1      : 0.3807


,fold,macro_auc_ovr,balanced_accuracy,macro_f1
0,1,0.592593,0.444444,0.392344
1,2,0.650118,0.425926,0.378337
2,3,0.669460,0.485185,0.460317
3,4,0.534061,0.375000,0.342105
4,5,0.630556,0.350000,0.300095



Confusion matrix:
[[ 0 15  6]
 [ 2 33 14]
 [ 0 18 25]]

Classification report:
                  precision    recall  f1-score   support

         Control     0.0000    0.0000    0.0000        21
Elevated PSA/Bx-     0.5000    0.6735    0.5739        49
 Prostate Cancer     0.5556    0.5814    0.5682        43

        accuracy                         0.5133       113
       macro avg     0.3519    0.4183    0.3807       113
    weighted avg     0.4282    0.5133    0.4651       113



In [ ]:
class_auc = roc_auc_score(
    y_3class,
    oof_3class,
    multi_class="ovr",
    average=None,
    labels=np.arange(N_CLASSES),
)

class_auc_df = pd.DataFrame({
    "class_id": np.arange(N_CLASSES),
    "class_name": CLASS_NAMES,
    "ovr_auc": class_auc,
    "subjects": np.bincount(y_3class),
})

display(class_auc_df)

,class_id,class_name,ovr_auc,subjects
0,0,Control,0.575569,21
1,1,Elevated PSA/Bx-,0.598214,49
2,2,Prostate Cancer,0.704319,43


### Peak registry and baseline diagnostics


In [ ]:
def _estimate_noise_sigma(values: np.ndarray) -> float:
    values_array = np.asarray(values, dtype=np.float64)
    differences = np.diff(values_array)
    if len(differences) == 0:
        return float(np.finfo(float).eps)

    centered = differences - float(np.median(differences))
    sigma = 1.4826 * float(np.median(np.abs(centered))) / np.sqrt(2.0)
    return max(sigma, float(np.finfo(float).eps))


def _asls_baseline(
    values: np.ndarray,
    lam: float = BASELINE_LAMBDA,
    p: float = BASELINE_ASYMMETRY,
    niter: int = 10,
) -> np.ndarray:
    values_array = np.asarray(values, dtype=np.float64)
    length = len(values_array)
    difference = sparse.diags(
        [1.0, -2.0, 1.0],
        [0, -1, -2],
        shape=(length, length - 2),
        dtype=float,
        format="csc",
    )
    penalty = (lam * difference.dot(difference.T)).tocsc()
    weights = np.ones(length, dtype=float)
    for _ in range(niter):
        weighted = sparse.spdiags(
            weights,
            0,
            length,
            length,
        ).tocsc()
        baseline = spsolve(
            (weighted + penalty).tocsc(),
            weights * values_array,
        )
        weights = p * (values_array > baseline) + (1.0 - p) * (values_array < baseline)
    return np.asarray(baseline, dtype=np.float64)


def _baseline_area_normalize(
    spectra: np.ndarray,
    grid: np.ndarray | None = None,
) -> np.ndarray:
    values = np.asarray(spectra, dtype=np.float64)
    matrix = values[None, :] if values.ndim == 1 else values
    integration_grid = np.asarray(
        target_grid if grid is None else grid,
        dtype=np.float64,
    )
    if len(integration_grid) != matrix.shape[1]:
        raise ValueError("spectrum and integration grid lengths differ")

    processed = np.empty_like(matrix, dtype=np.float64)
    for index, row in enumerate(matrix):
        corrected = np.clip(
            row - _asls_baseline(row),
            0.0,
            None,
        )
        area = np.trapezoid(corrected, integration_grid)
        processed[index] = corrected / area if area > 0.0 else corrected
    return processed[0] if values.ndim == 1 else processed

source_grid = np.asarray(target_grid, dtype=np.float64)
target_grid = STK_TARGET_GRID.copy()
GRID_STEP_CM1 = float(np.median(np.diff(target_grid)))
RESOLUTION_MIN_DISTANCE_POINTS = max(
    1,
    int(np.ceil(INSTRUMENT_RESOLUTION_FWHM_CM1 / GRID_STEP_CM1)),
)

X = stk_v2_peak_input(
    X_RAW_SUBJECT_MEAN,
    source_grid=source_grid,
    target_grid=target_grid,
)

def merge_peak_centers(
    centers: np.ndarray,
    heights: np.ndarray,
    tolerance_cm1: float = PEAK_MATCH_TOLERANCE_CM1,
) -> np.ndarray:
    centers_array = np.asarray(centers, dtype=np.float64)
    heights_array = np.asarray(heights, dtype=np.float64)

    if centers_array.shape != heights_array.shape:
        raise ValueError("centers와 heights의 shape이 같아야 합니다")
    if tolerance_cm1 < 0:
        raise ValueError("tolerance_cm1 must be non-negative")

    order = np.argsort(centers_array)
    sorted_centers = centers_array[order]
    sorted_heights = heights_array[order]

    if len(sorted_centers) == 0:
        return sorted_centers
    if tolerance_cm1 == 0:
        return np.unique(sorted_centers)

    groups: list[list[int]] = [[0]]
    for index, center in enumerate(sorted_centers[1:], start=1):
        group_start = groups[-1][0]
        group_minimum = float(sorted_centers[group_start])

        # 그룹의 전체 범위가 tolerance 이내여야 같은 peak로 묶는다.
        if float(center) - group_minimum <= tolerance_cm1:
            groups[-1].append(index)
        else:
            groups.append([index])

    # 각 그룹에서 가장 높은 실제 검출 중심을 대표값으로 선택한다.
    return np.asarray(
        [
            sorted_centers[group[int(np.argmax(sorted_heights[group]))]]
            for group in groups
        ],
        dtype=np.float64,
    )


def _voigt_sum(
    x: np.ndarray,
    baseline: float,
    slope: float,
    *params: float,
) -> np.ndarray:
    fitted = baseline + slope * (x - float(np.mean(x)))
    for area, center, sigma, gamma in np.asarray(params).reshape(-1, 4):
        fitted = fitted + area * voigt_profile(x - center, sigma, gamma)
    return fitted


def _voigt_fwhm(sigma: float, gamma: float) -> float:
    fwhm_gaussian = 2.0 * sigma * np.sqrt(2.0 * np.log(2.0))
    fwhm_lorentzian = 2.0 * gamma
    return float(
        0.5346 * fwhm_lorentzian
        + np.sqrt(
            0.2166 * fwhm_lorentzian**2
            + fwhm_gaussian**2
        )
    )


def _initial_center_guesses(
    x: np.ndarray,
    y_values: np.ndarray,
    component_count: int,
) -> list[float]:
    if component_count not in {1, 2}:
        raise ValueError("Only one- and two-component models are supported")

    step = float(np.median(np.diff(x)))
    noise_sigma = _estimate_noise_sigma(y_values)
    prominence = max(
        3.0 * noise_sigma,
        float(np.ptp(y_values)) * 0.01,
        np.finfo(float).eps,
    )
    peak_indices, properties = find_peaks(
        y_values,
        prominence=prominence,
        distance=max(
            1,
            int(np.ceil(INSTRUMENT_RESOLUTION_FWHM_CM1 / step)),
        ),
    )
    ranked = peak_indices[
        np.argsort(
            properties.get("prominences", np.asarray([]))
        )[::-1]
    ]

    if component_count == 1:
        return [
            float(x[ranked[0]])
            if len(ranked)
            else float(x[np.argmax(y_values)])
        ]

    if len(ranked) >= 2:
        return sorted(float(x[index]) for index in ranked[:2])

    center = (
        float(x[ranked[0]])
        if len(ranked)
        else float(x[np.argmax(y_values)])
    )
    offset = max(
        1.5 * INSTRUMENT_RESOLUTION_FWHM_CM1,
        2.0 * step,
    )
    return [
        max(float(x[0]), center - offset),
        min(float(x[-1]), center + offset),
    ]


def _fit_voigt_count(
    x: np.ndarray,
    y_values: np.ndarray,
    component_count: int,
) -> dict[str, object]:
    x_array = np.asarray(x, dtype=np.float64)
    y_array = np.asarray(y_values, dtype=np.float64)

    if len(x_array) != len(y_array):
        raise ValueError("x와 y_values의 길이가 같아야 합니다")
    if len(x_array) < 12:
        raise ValueError("Peak region needs at least 12 grid points")
    if np.any(np.diff(x_array) <= 0):
        raise ValueError("x는 오름차순이어야 합니다")

    step = float(np.median(np.diff(x_array)))
    center_guesses = _initial_center_guesses(
        x_array,
        y_array,
        component_count,
    )

    baseline_guess = float(np.quantile(y_array, 0.1))
    width_guess = max(
        2.0 * step,
        0.5 * INSTRUMENT_RESOLUTION_FWHM_CM1,
    )
    area_guess = max(
        float(np.ptp(y_array)) * width_guess,
        np.finfo(float).eps,
    )

    lower = [-np.inf, -np.inf]
    upper = [np.inf, np.inf]
    initial = [baseline_guess, 0.0]

    for center in center_guesses:
        initial.extend([area_guess, center, width_guess, width_guess])
        lower.extend([
            0.0,
            float(x_array[0]),
            max(step / 4.0, 0.1),
            max(step / 4.0, 0.1),
        ])
        upper.extend([
            np.inf,
            float(x_array[-1]),
            float(np.ptp(x_array)),
            float(np.ptp(x_array)),
        ])

    parameters, covariance = curve_fit(
        _voigt_sum,
        x_array,
        y_array,
        p0=initial,
        bounds=(lower, upper),
        maxfev=20_000,
    )

    predicted = _voigt_sum(x_array, *parameters)
    residual = y_array - predicted
    residual_sum = max(
        float(np.sum(residual**2)),
        np.finfo(float).tiny,
    )

    sample_count = len(x_array)
    parameter_count = len(parameters)
    rmse = float(np.sqrt(residual_sum / sample_count))
    log_likelihood_term = sample_count * np.log(
        residual_sum / sample_count
    )
    aic = float(log_likelihood_term + 2.0 * parameter_count)
    bic = float(
        log_likelihood_term
        + parameter_count * np.log(sample_count)
    )
    aicc = (
        float(
            aic
            + 2.0
            * parameter_count
            * (parameter_count + 1)
            / (sample_count - parameter_count - 1)
        )
        if sample_count > parameter_count + 1
        else float("inf")
    )

    covariance_diagonal = np.diag(covariance)
    parameter_standard_errors = np.sqrt(
        np.maximum(covariance_diagonal, 0.0)
    )
    components = np.asarray(
        parameters[2:],
        dtype=np.float64,
    ).reshape(-1, 4)
    component_standard_errors = np.asarray(
        parameter_standard_errors[2:],
        dtype=np.float64,
    ).reshape(-1, 4)

    residual_noise_sigma = _estimate_noise_sigma(residual)
    residual_peak_indices, _ = find_peaks(
        residual,
        prominence=max(
            3.0 * residual_noise_sigma,
            np.finfo(float).eps,
        ),
        distance=max(
            1,
            int(np.ceil(INSTRUMENT_RESOLUTION_FWHM_CM1 / step)),
        ),
    )

    return {
        "parameters": parameters,
        "covariance": covariance,
        "parameter_standard_errors": parameter_standard_errors,
        "components": components,
        "component_standard_errors": component_standard_errors,
        "predicted": predicted,
        "residual": residual,
        "rss": residual_sum,
        "rmse": rmse,
        "aic": aic,
        "aicc": aicc,
        "bic": bic,
        "residual_noise_sigma": residual_noise_sigma,
        "residual_peak_count": int(len(residual_peak_indices)),
    }


def decompose_peak_region(
    x: np.ndarray,
    y_values: np.ndarray,
    left_cm1: float,
    right_cm1: float,
) -> tuple[pd.DataFrame, dict[str, object]]:
    mask = (x >= left_cm1) & (x <= right_cm1)
    x_region = np.asarray(x[mask], dtype=np.float64)
    y_region = np.asarray(y_values[mask], dtype=np.float64)

    if len(x_region) < 12:
        raise ValueError("Peak region needs at least 12 grid points")

    model_results = {
        component_count: _fit_voigt_count(
            x_region,
            y_region,
            component_count,
        )
        for component_count in range(
            1,
            DECONVOLUTION_MAX_COMPONENTS + 1,
        )
    }

    selected_count = 1
    delta_bic_by_count: dict[int, float] = {}
    for component_count in range(2, DECONVOLUTION_MAX_COMPONENTS + 1):
        previous_bic = float(
            model_results[component_count - 1]["bic"]
        )
        current_bic = float(
            model_results[component_count]["bic"]
        )
        delta_bic = previous_bic - current_bic
        delta_bic_by_count[component_count] = delta_bic

        if delta_bic >= DECONVOLUTION_BIC_IMPROVEMENT:
            selected_count = component_count
        else:
            break

    selected = model_results[selected_count]
    components = np.asarray(selected["components"])
    component_errors = np.asarray(
        selected["component_standard_errors"]
    )

    ordered_components = components[
        np.argsort(components[:, 1])
    ]
    ordered_errors = component_errors[
        np.argsort(components[:, 1])
    ]

    centers = ordered_components[:, 1]
    gaps = np.diff(centers)
    fwhms = np.asarray([
        _voigt_fwhm(float(sigma), float(gamma))
        for _, _, sigma, gamma in ordered_components
    ])
    areas = ordered_components[:, 0]
    mean_fitted_fwhm = (
        float(np.mean(fwhms)) if len(fwhms) else float("nan")
    )
    pairwise_fwhm = (
        (fwhms[:-1] + fwhms[1:]) / 2.0
        if len(gaps)
        else np.asarray([], dtype=np.float64)
    )

    n_instrument_resolved_peaks = (
        1 + int(
            np.sum(
                (gaps >= INSTRUMENT_RESOLUTION_FWHM_CM1)
                & (gaps >= pairwise_fwhm)
            )
        )
        if len(ordered_components)
        else 0
    )

    minimum_center_gap = (
        float(np.min(gaps))
        if len(gaps)
        else float("nan")
    )

    resolution_gap_threshold = max(
        INSTRUMENT_RESOLUTION_FWHM_CM1,
        mean_fitted_fwhm,
    )
    if selected_count == 1:
        resolution_status = "single_component"
    elif minimum_center_gap < resolution_gap_threshold:
        resolution_status = "unresolved"
    elif (
        minimum_center_gap
        <= RESOLUTION_BORDERLINE_FACTOR * resolution_gap_threshold
    ):
        resolution_status = "borderline"
    else:
        resolution_status = "potentially_resolved"

    minimum_area_fraction = (
        float(np.min(areas) / np.sum(areas))
        if len(areas) > 1 and np.sum(areas) > 0
        else 1.0
    )
    separation_ratio = (
        float(minimum_center_gap / np.mean(fwhms))
        if len(gaps) and np.mean(fwhms) > 0
        else float("nan")
    )
    noise_sigma = _estimate_noise_sigma(y_region)

    rows = []
    for component_index, (
        component,
        errors,
    ) in enumerate(
        zip(ordered_components, ordered_errors, strict=True),
        start=1,
    ):
        area, center, sigma, gamma = component
        area_error, center_error, sigma_error, gamma_error = errors
        rows.append({
            "component": component_index,
            "center_cm-1": float(center),
            "center_se_cm-1": float(center_error),
            "area": float(area),
            "area_se": float(area_error),
            "sigma_cm-1": float(sigma),
            "sigma_se_cm-1": float(sigma_error),
            "gamma_cm-1": float(gamma),
            "gamma_se_cm-1": float(gamma_error),
            "fwhm_cm-1": _voigt_fwhm(float(sigma), float(gamma)),
        })

    model_metrics = pd.DataFrame([
        {
            "components": component_count,
            "rss": float(result["rss"]),
            "rmse": float(result["rmse"]),
            "aic": float(result["aic"]),
            "aicc": float(result["aicc"]),
            "bic": float(result["bic"]),
            "delta_bic_from_previous": delta_bic_by_count.get(
                component_count,
                float("nan"),
            ),
        }
        for component_count, result in model_results.items()
    ])

    diagnostics = {
        "selected_components": selected_count,
        "n_deconvolved_components": selected_count,
        "n_instrument_resolved_peaks": n_instrument_resolved_peaks,
        "mean_fitted_fwhm_cm-1": mean_fitted_fwhm,
        "resolution_gap_threshold_cm-1": resolution_gap_threshold,
        "resolution_status": resolution_status,
        "resolution_class": resolution_status,
        "bic_one": float(model_results[1]["bic"]),
        "bic_selected": float(selected["bic"]),
        "bic_improvement": float(
            model_results[1]["bic"] - selected["bic"]
        ),
        "rmse": float(selected["rmse"]),
        "rmse_over_noise": float(
            selected["rmse"] / max(noise_sigma, np.finfo(float).eps)
        ),
        "aic": float(selected["aic"]),
        "aicc": float(selected["aicc"]),
        "minimum_center_gap_cm-1": minimum_center_gap,
        "separation_ratio": separation_ratio,
        "minimum_area_fraction": minimum_area_fraction,
        "residual_peak_count": int(selected["residual_peak_count"]),
        "instrument_resolution_fwhm_cm-1": (
            INSTRUMENT_RESOLUTION_FWHM_CM1
        ),
        "model_metrics": model_metrics,
        "x_region": x_region,
        "y_region": y_region,
        "predicted": selected["predicted"],
        "residual": selected["residual"],
    }

    return pd.DataFrame(rows), diagnostics


def detect_peak_centers(
    spectrum: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Smoothing 없이 후보 중심과 원본 intensity를 반환한다."""

    spectrum_array = np.asarray(spectrum, dtype=np.float64)
    noise_sigma = _estimate_noise_sigma(spectrum_array)
    peak_indices, _ = find_peaks(
        spectrum_array,
        prominence=max(
            3.0 * noise_sigma,
            np.finfo(float).eps,
        ),
        distance=RESOLUTION_MIN_DISTANCE_POINTS,
    )

    return (
        np.asarray(target_grid[peak_indices], dtype=np.float64),
        np.asarray(spectrum_array[peak_indices], dtype=np.float64),
    )


subject_detections = [
    detect_peak_centers(spectrum)
    for spectrum in X
]
subject_peak_centers = [
    centers
    for centers, _ in subject_detections
]
nonempty_detections = [
    (centers, heights)
    for centers, heights in subject_detections
    if len(centers)
]

if nonempty_detections:
    detected_centers = np.concatenate([
        centers
        for centers, _ in nonempty_detections
    ])
    detected_heights = np.concatenate([
        heights
        for _, heights in nonempty_detections
    ])
else:
    detected_centers = np.asarray([], dtype=np.float64)
    detected_heights = np.asarray([], dtype=np.float64)

matched_centers = merge_peak_centers(
    detected_centers,
    detected_heights,
)
peak_registry = pd.DataFrame({
    "center_cm-1": matched_centers,
})
peak_registry["subject_count"] = [
    sum(
        np.any(
            np.abs(centers - center)
            <= PEAK_MATCH_TOLERANCE_CM1
        )
        for centers in subject_peak_centers
    )
    for center in matched_centers
]
peak_registry["reproducibility"] = (
    peak_registry["subject_count"]
    / len(subject_peak_centers)
)
peak_registry = peak_registry.sort_values(
    ["reproducibility", "center_cm-1"],
    ascending=[False, True],
).reset_index(drop=True)

print({
    "grid_step_cm-1": float(
        np.median(np.diff(target_grid))
    ),
    "instrument_resolution_fwhm_cm-1": (
        INSTRUMENT_RESOLUTION_FWHM_CM1
    ),
    "peak_input": "AsLS baseline-corrected + area-normalized; no smoothing",
    "smoothing": "disabled",
    "minimum_peak_distance_cm-1": (
        RESOLUTION_MIN_DISTANCE_POINTS * GRID_STEP_CM1
    ),
    "detected_candidate_peaks": len(detected_centers),
    "matched_candidate_peaks": len(matched_centers),
    "matching_tolerance_cm-1": PEAK_MATCH_TOLERANCE_CM1,
})
peak_registry.to_csv(
    NOTEBOOK_OUTPUT_DIR / "peak_registry.csv",
    index=False,
    encoding="utf-8-sig",
)
display(peak_registry.head(20))


{'grid_step_cm-1': 1.9229122055673997, 'instrument_resolution_fwhm_cm-1': 2.0, 'peak_input': 'AsLS baseline-corrected + area-normalized; no smoothing', 'smoothing': 'disabled', 'minimum_peak_distance_cm-1': 3.862660944206027, 'detected_candidate_peaks': 2225, 'matched_candidate_peaks': 373, 'matching_tolerance_cm-1': 0.0}


,center_cm-1,subject_count,reproducibility
0,723.126338,68,0.601770
1,1146.167024,65,0.575221
2,1001.948608,49,0.433628
3,936.569593,47,0.415929
4,846.192719,45,0.398230
5,684.668094,41,0.362832
6,934.646681,40,0.353982
7,686.591006,39,0.345133
8,848.115632,38,0.336283
9,892.342612,38,0.336283


In [ ]:
"""
AsLS baseline correction diagnostics for the AECD SERS cohort.

WHY
---
Analysis C's "top discriminative peaks" clustered in 1900-2100 cm-1 -- the
biological Raman SILENT REGION where real analyte peaks (Phe 1001, amide I
1650, CH2 1450) should NOT dominate. That is a red flag that AsLS baseline
correction left substrate/residual structure behind (or distorted peaks).
This module diagnoses baseline quality BEFORE trusting any peak result.

What it checks
--------------
1) lam SWEEP: fit AsLS at several lam values on representative spectra and
   overlay raw + fitted baseline + corrected -> pick a lam that removes the
   broad background WITHOUT eating the fingerprint peaks.
2) BEFORE/AFTER per class: class-mean raw vs corrected, so you can see whether
   correction changed class separation or introduced artefacts.
3) SILENT-REGION residual metric: after correction, the silent region
   (default 1800-2200 cm-1) should be ~flat/zero. We quantify residual energy
   there vs the fingerprint region. High silent-region energy => baseline
   under-correction => the suspicious peaks are artefacts.
4) NEGATIVE-fraction & clipping loss: how much signal is being clipped to 0
   (a sign lam/p are off).

INPUTS (notebook namespace)
---------------------------
    aligned, metadata, subject_keys, source_grid, target_grid
    TARGET_COLUMN (default "cohort_group"), RANDOM_STATE, QC_MAD_THRESHOLD
Outputs PNG/CSV to OUTPUT_DIR.
"""

import os
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.linalg import spsolve
from scipy.integrate import trapezoid

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


OUTPUT_DIR = os.environ.get(
    "AECD_OUTDIR",
    str(globals().get("NOTEBOOK_OUTPUT_DIR",
                      os.path.join(os.getcwd(), "aecd_baseline_diagnostics"))),
)
os.makedirs(OUTPUT_DIR, exist_ok=True)
QC_MAD_THRESHOLD = float(globals().get("QC_MAD_THRESHOLD", 3.5))
TARGET_COLUMN = str(globals().get("TARGET_COLUMN", "cohort_group"))

RAW_CONTROL = "control"
RAW_PDC = "prostate disease control"
RAW_CANCER = "prostate"
NICE = {RAW_CONTROL: "Control", RAW_PDC: "Elevated PSA/Bx-",
        RAW_CANCER: "Prostate Cancer"}

# regions (cm-1)
FINGERPRINT = (400, 1800)
SILENT = (1800, 2200)          # should be flat after good baseline removal
LAM_SWEEP = (1e4, 1e5, 1e6, 1e7)
P_ASYM = 0.01


# ------------------------------------------------------------------
# QC subject means (same logic as the main suite)
# ------------------------------------------------------------------
def _qc_mean(reps, mad_threshold=QC_MAD_THRESHOLD):
    reps = np.asarray(reps, dtype=np.float64)
    if reps.ndim != 2 or len(reps) < 2:
        return reps.reshape(-1, reps.shape[-1]).mean(0)
    med = np.median(reps, 0)
    resid = reps - med
    scale = np.maximum(np.median(np.abs(resid), 0), np.finfo(float).eps)
    dist = np.sqrt(np.mean((resid / scale) ** 2, 1))
    dmed = float(np.median(dist))
    dmad = float(np.median(np.abs(dist - dmed)))
    limit = (dmed + 3.0 * float(np.std(dist))) if dmad == 0 \
        else (dmed + mad_threshold * 1.4826 * dmad)
    keep = dist <= max(limit, dmed)
    if keep.sum() < 2:
        keep = np.zeros(len(dist), bool)
        keep[np.argsort(dist)[:2]] = True
    return reps[keep].mean(0)


def build_qc_matrix():
    aligned = np.asarray(globals()["aligned"], dtype=np.float64)
    metadata = globals()["metadata"]
    subj_order = list(globals()["subject_keys"])
    subj_col = metadata["subject_key"].to_numpy()
    label_col = metadata[TARGET_COLUMN].astype(str).to_numpy()
    Xr, yr = [], []
    for k in subj_order:
        m = subj_col == k
        if not m.any():
            continue
        Xr.append(_qc_mean(aligned[m]))
        yr.append(pd.Series(label_col[m]).mode().iat[0])
    return np.vstack(Xr).astype(np.float64), np.asarray(yr).astype(str)


# ------------------------------------------------------------------
# AsLS
# ------------------------------------------------------------------
def asls_baseline(y, lam=1e5, p=0.01, niter=10):
    L = len(y)
    D = sparse.diags([1.0, -2.0, 1.0], [0, -1, -2],
                     shape=(L, L - 2), format="csc")
    D = lam * D.dot(D.T)
    w = np.ones(L)
    for _ in range(niter):
        W = sparse.spdiags(w, 0, L, L).tocsc()
        z = spsolve((W + D).tocsc(), w * y)
        w = p * (y > z) + (1 - p) * (y < z)
    return z


def _region_mask(grid, lo, hi):
    return (grid >= lo) & (grid < hi)


def _corrected(y, lam, p):
    base = asls_baseline(y, lam=lam, p=p)
    corr = np.clip(y - base, 0, None)
    return base, corr


# ------------------------------------------------------------------
# (1) lam sweep on representative spectra
# ------------------------------------------------------------------
def diagnose_lam_sweep(Xqc, y_fine, grid, lam_sweep=LAM_SWEEP, p=P_ASYM):
    # one representative (median-energy) subject per class
    reps = {}
    for c in (RAW_CONTROL, RAW_PDC, RAW_CANCER):
        idx = np.where(y_fine == c)[0]
        energies = Xqc[idx].sum(1)
        reps[c] = idx[np.argsort(energies)[len(idx) // 2]]

    fig, axes = plt.subplots(
        len(reps), len(lam_sweep),
        figsize=(3.2 * len(lam_sweep), 2.6 * len(reps)),
        sharex=True,
    )
    sil = _region_mask(grid, *SILENT)
    for r, (c, si) in enumerate(reps.items()):
        y = Xqc[si]
        for col, lam in enumerate(lam_sweep):
            ax = axes[r, col]
            base, corr = _corrected(y, lam, p)
            ax.plot(grid, y, color="0.6", lw=0.7, label="raw")
            ax.plot(grid, base, color="#d7191c", lw=1.0, label="baseline")
            ax.plot(grid, corr, color="#2c7fb8", lw=0.8, label="corrected")
            ax.axvspan(*SILENT, color="orange", alpha=0.10)
            # silent-region residual energy fraction
            sil_frac = (trapezoid(corr[sil], grid[sil]) /
                        max(trapezoid(corr, grid), 1e-12))
            ax.set_title(f"{NICE[c]}\nlam={lam:.0e}  silent={sil_frac:.1%}",
                         fontsize=8)
            if r == 0 and col == 0:
                ax.legend(fontsize=6, loc="upper right")
            ax.tick_params(labelsize=6)
    fig.suptitle("AsLS lam sweep — raw / baseline / corrected "
                 "(orange = silent region 1800–2200)", fontsize=11)
    fig.supxlabel("Raman shift (cm$^{-1}$)")
    fig.tight_layout(rect=(0, 0.02, 1, 0.97))
    path = os.path.join(OUTPUT_DIR, "diag1_lam_sweep.png")
    fig.savefig(path, dpi=180); plt.close(fig)
    print(f"[1] lam sweep figure -> {path}")


# ------------------------------------------------------------------
# (2) before/after class means + (3) silent-region metric table
# ------------------------------------------------------------------
def diagnose_before_after(Xqc, y_fine, grid, lam=1e5, p=P_ASYM):
    fp = _region_mask(grid, *FINGERPRINT)
    sil = _region_mask(grid, *SILENT)

    # corrected matrix
    Xc = np.empty_like(Xqc)
    clip_loss = np.empty(len(Xqc))
    for i, y in enumerate(Xqc):
        base, corr = _corrected(y, lam, p)
        Xc[i] = corr
        raw_pos = np.clip(y - base, None, None)
        clip_loss[i] = np.mean((y - base) < 0)     # fraction of points clipped

    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                             gridspec_kw={"hspace": 0.15})
    colors = {RAW_CONTROL: "#2c7fb8", RAW_PDC: "#f0a202", RAW_CANCER: "#d7191c"}
    for c in (RAW_CONTROL, RAW_PDC, RAW_CANCER):
        m = y_fine == c
        axes[0].plot(grid, Xqc[m].mean(0), lw=1.2, color=colors[c],
                     label=NICE[c])
        axes[1].plot(grid, Xc[m].mean(0), lw=1.2, color=colors[c],
                     label=NICE[c])
    for ax in axes:
        ax.axvspan(*SILENT, color="orange", alpha=0.10)
        ax.grid(alpha=0.15)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    axes[0].set_title(f"A. RAW class-mean (QC)  |  B. AsLS-corrected "
                      f"(lam={lam:.0e}, p={p})", loc="left", fontweight="bold")
    axes[0].set_ylabel("raw intensity"); axes[1].set_ylabel("corrected")
    axes[1].set_xlabel("Raman shift (cm$^{-1}$)")
    axes[0].legend(frameon=False, fontsize=8)
    path = os.path.join(OUTPUT_DIR, "diag2_before_after.png")
    fig.savefig(path, dpi=180, bbox_inches="tight"); plt.close(fig)
    print(f"[2] before/after figure -> {path}")

    # per-class silent-region residual metric
    rows = []
    for c in (RAW_CONTROL, RAW_PDC, RAW_CANCER):
        m = y_fine == c
        cc = Xc[m]
        sil_energy = trapezoid(cc[:, sil], grid[sil], axis=1)
        fp_energy = trapezoid(cc[:, fp], grid[fp], axis=1)
        tot = np.clip(sil_energy + fp_energy, 1e-12, None)
        rows.append({
            "class": NICE[c],
            "silent_energy_frac_mean": float(np.mean(sil_energy / tot)),
            "fingerprint_energy_frac_mean": float(np.mean(fp_energy / tot)),
            "clip_loss_frac_mean": float(np.mean(clip_loss[m])),
        })
    tbl = pd.DataFrame(rows)
    tbl.to_csv(os.path.join(OUTPUT_DIR, "diag3_silent_region_metrics.csv"),
               index=False, encoding="utf-8-sig")

    print("\n[3] Silent-region residual metrics (lam={:.0e})".format(lam))
    print(tbl.to_string(index=False))
    print("\n  Interpretation:")
    worst = tbl["silent_energy_frac_mean"].max()
    if worst > 0.15:
        print(f"  ⚠ silent-region energy up to {worst:.1%} -> UNDER-CORRECTION."
              f" Increase lam (try 1e6–1e7) or mask 1800–2200 before modeling.")
    else:
        print(f"  ✓ silent-region energy low ({worst:.1%}); baseline looks OK."
              f" Suspicious peaks may instead be true (check band assignments).")
    return tbl


# ------------------------------------------------------------------
# (4) silent vs fingerprint energy across lam (global)
# ------------------------------------------------------------------
def diagnose_lam_energy_curve(Xqc, grid, lam_sweep=LAM_SWEEP, p=P_ASYM):
    sil = _region_mask(grid, *SILENT)
    fp = _region_mask(grid, *FINGERPRINT)
    rows = []
    for lam in lam_sweep:
        sfrac, closs = [], []
        for y in Xqc:
            base, corr = _corrected(y, lam, p)
            se = trapezoid(corr[sil], grid[sil])
            fe = trapezoid(corr[fp], grid[fp])
            sfrac.append(se / max(se + fe, 1e-12))
            closs.append(np.mean((y - base) < 0))
        rows.append({"lam": lam,
                     "silent_frac_mean": float(np.mean(sfrac)),
                     "clip_loss_mean": float(np.mean(closs))})
    tbl = pd.DataFrame(rows)
    tbl.to_csv(os.path.join(OUTPUT_DIR, "diag4_lam_energy_curve.csv"),
               index=False)

    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.plot(tbl["lam"], tbl["silent_frac_mean"], "o-",
            color="#d7191c", label="silent-region energy fraction")
    ax.plot(tbl["lam"], tbl["clip_loss_mean"], "s--",
            color="#2c7fb8", label="clipped-point fraction")
    ax.set_xscale("log"); ax.set_xlabel("AsLS lam")
    ax.set_ylabel("fraction"); ax.grid(alpha=0.3)
    ax.axhspan(0, 0.15, color="green", alpha=0.06)
    ax.set_title("Choose lam: low silent-energy AND low clipping")
    ax.legend(fontsize=8)
    path = os.path.join(OUTPUT_DIR, "diag4_lam_energy_curve.png")
    fig.tight_layout(); fig.savefig(path, dpi=180); plt.close(fig)
    print(f"\n[4] lam energy curve -> {path}")
    print(tbl.to_string(index=False))

    best = tbl.loc[tbl["silent_frac_mean"].idxmin()]
    print(f"\n  -> lam minimizing silent-region energy: {best['lam']:.0e} "
          f"(silent={best['silent_frac_mean']:.1%}, "
          f"clip={best['clip_loss_mean']:.1%})")
    return tbl


# ------------------------------------------------------------------
# run
# ------------------------------------------------------------------
def run_baseline_diagnostics(lam_main=1e5, lam_sweep=LAM_SWEEP, p=P_ASYM):
    print("Building QC subject-mean matrix ...")
    Xqc, y_fine = build_qc_matrix()
    grid = np.asarray(
        globals().get("source_grid", globals()["target_grid"]),
        dtype=np.float64,
    )
    print(f"  X_qc={Xqc.shape}  grid=[{grid.min():.0f},{grid.max():.0f}]")

    diagnose_lam_sweep(Xqc, y_fine, grid, lam_sweep, p)
    diagnose_before_after(Xqc, y_fine, grid, lam=lam_main, p=p)
    diagnose_lam_energy_curve(Xqc, grid, lam_sweep, p)

    print("\nSaved to:", os.path.abspath(OUTPUT_DIR))
    print("  diag1_lam_sweep.png")
    print("  diag2_before_after.png")
    print("  diag3_silent_region_metrics.csv")
    print("  diag4_lam_energy_curve.png / .csv")


if __name__ == "__main__" or True:
    run_baseline_diagnostics()


Building QC subject-mean matrix ...
  X_qc=(113, 933)  grid=[402,2198]


ValueError: x and y must have same first dimension, but have shapes (935,) and (933,)

In [ ]:
import matplotlib.pyplot as plt


# 분석할 region 중심이다. grid bin을 peak registry에서 임의 병합하지 않는다.
REQUESTED_RAW_CENTER_CM1 = 1001.5


def _subjects_with_raw_region(center_cm1: float) -> list[int]:
    left_cm1 = center_cm1 - DECONVOLUTION_HALF_WINDOW_CM1
    right_cm1 = center_cm1 + DECONVOLUTION_HALF_WINDOW_CM1
    region_mask = (target_grid >= left_cm1) & (target_grid <= right_cm1)
    return list(range(len(X))) if np.any(region_mask) else []


raw_candidate_subjects = _subjects_with_raw_region(
    REQUESTED_RAW_CENTER_CM1
)

ANALYZED_RAW_CENTER_CM1 = REQUESTED_RAW_CENTER_CM1

if not raw_candidate_subjects:
    raise RuntimeError(
        "No subject spectrum covers the selected peak region"
    )


def analyze_raw_region(center_cm1: float):
    candidate_subjects = _subjects_with_raw_region(
        center_cm1
    )

    fit_rows = []
    component_rows = []
    component_tables_by_subject = {}
    diagnostics_by_subject = {}

    for subject_index in candidate_subjects:
        try:
            component_table, diagnostics = (
                decompose_peak_region(
                    x=target_grid,
                    y_values=X[subject_index],
                    left_cm1=(
                        center_cm1
                        - DECONVOLUTION_HALF_WINDOW_CM1
                    ),
                    right_cm1=(
                        center_cm1
                        + DECONVOLUTION_HALF_WINDOW_CM1
                    ),
                )
            )
        except (RuntimeError, ValueError):
            continue

        component_tables_by_subject[subject_index] = (
            component_table
        )
        diagnostics_by_subject[subject_index] = diagnostics

        fit_rows.append({
            "registry_center_cm-1": center_cm1,
            "subject_index": subject_index,
            "candidate_subject_count": len(
                candidate_subjects
            ),
            "n_deconvolved_components": int(
                diagnostics["n_deconvolved_components"]
            ),
            "n_instrument_resolved_peaks": int(
                diagnostics["n_instrument_resolved_peaks"]
            ),
            "resolution_status": str(
                diagnostics["resolution_status"]
            ),
            "bic_improvement": float(
                diagnostics["bic_improvement"]
            ),
            "rmse": float(diagnostics["rmse"]),
            "rmse_over_noise": float(
                diagnostics["rmse_over_noise"]
            ),
            "minimum_center_gap_cm-1": float(
                diagnostics["minimum_center_gap_cm-1"]
            ),
            "mean_fitted_fwhm_cm-1": float(
                diagnostics["mean_fitted_fwhm_cm-1"]
            ),
            "resolution_gap_threshold_cm-1": float(
                diagnostics["resolution_gap_threshold_cm-1"]
            ),
            "residual_peak_count": int(
                diagnostics["residual_peak_count"]
            ),
        })

        for component_row in component_table.to_dict(
            orient="records"
        ):
            component_rows.append({
                "registry_center_cm-1": center_cm1,
                "subject_index": subject_index,
                **component_row,
                "resolution_status": str(
                    diagnostics["resolution_status"]
                ),
                "bic_improvement": float(
                    diagnostics["bic_improvement"]
                ),
                "rmse": float(diagnostics["rmse"]),
            })

    fit_results = pd.DataFrame(fit_rows)
    component_results = pd.DataFrame(component_rows)

    if fit_results.empty:
        raise RuntimeError(
            "No raw spectrum could be fitted in this region"
        )

    return (
        fit_results,
        component_results,
        component_tables_by_subject,
        diagnostics_by_subject,
    )


(
    raw_deconvolution_results,
    raw_component_results,
    raw_component_tables,
    raw_diagnostics,
) = analyze_raw_region(
    ANALYZED_RAW_CENTER_CM1
)

raw_region_summary = pd.DataFrame([{
    "analyzed_center_cm-1": ANALYZED_RAW_CENTER_CM1,
    "candidate_subject_count": int(
        raw_deconvolution_results[
            "candidate_subject_count"
        ].iloc[0]
    ),
    "fitted_subject_count": len(
        raw_deconvolution_results
    ),
    "one_component_subjects": int(
        np.sum(
            raw_deconvolution_results[
                "n_deconvolved_components"
            ]
            == 1
        )
    ),
    "two_component_subjects": int(
        np.sum(
            raw_deconvolution_results[
                "n_deconvolved_components"
            ]
            == 2
        )
    ),
    "potentially_resolved_fraction": float(
        np.mean(
            raw_deconvolution_results[
                "resolution_status"
            ]
            .isin(["potentially_resolved"])
        )
    ),
    "median_bic_improvement": float(
        raw_deconvolution_results[
            "bic_improvement"
        ].median()
    ),
    "median_rmse": float(
        raw_deconvolution_results["rmse"].median()
    ),
    "median_rmse_over_noise": float(
        raw_deconvolution_results[
            "rmse_over_noise"
        ].median()
    ),
    "median_center_gap_cm-1": float(
        raw_deconvolution_results[
            "minimum_center_gap_cm-1"
        ].median()
    ),
}])


def plot_voigt_decomposition(
    components: pd.DataFrame,
    diagnostics: dict[str, object],
    title: str,
):
    x_region = np.asarray(
        diagnostics["x_region"],
        dtype=np.float64,
    )
    y_region = np.asarray(
        diagnostics["y_region"],
        dtype=np.float64,
    )
    predicted = np.asarray(
        diagnostics["predicted"],
        dtype=np.float64,
    )
    residual = np.asarray(
        diagnostics["residual"],
        dtype=np.float64,
    )

    x_plot = np.linspace(
        float(x_region[0]),
        float(x_region[-1]),
        1000,
    )

    component_curves = []
    raw_component_sum = np.zeros_like(
        x_region,
        dtype=np.float64,
    )

    for row in components.to_dict(orient="records"):
        component_curve = (
            float(row["area"])
            * voigt_profile(
                x_plot - float(row["center_cm-1"]),
                float(row["sigma_cm-1"]),
                float(row["gamma_cm-1"]),
            )
        )
        component_curves.append(component_curve)

        raw_component_sum += (
            float(row["area"])
            * voigt_profile(
                x_region - float(row["center_cm-1"]),
                float(row["sigma_cm-1"]),
                float(row["gamma_cm-1"]),
            )
        )

    baseline_raw = predicted - raw_component_sum
    baseline_coefficients = np.polyfit(
        x_region - np.mean(x_region),
        baseline_raw,
        1,
    )
    baseline = (
        baseline_coefficients[1]
        + baseline_coefficients[0]
        * (x_plot - np.mean(x_region))
    )
    total_fit = baseline + np.sum(
        np.vstack(component_curves),
        axis=0,
    )

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(11, 7),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1]},
    )
    fit_axis, residual_axis = axes

    fit_axis.scatter(
        x_region,
        y_region,
        color="black",
        s=20,
        label="baseline + area-normalized X (no smoothing)",
        zorder=3,
    )
    fit_axis.plot(
        x_plot,
        total_fit,
        color="tab:red",
        linewidth=2.2,
        label="total Voigt fit",
    )
    fit_axis.plot(
        x_plot,
        baseline,
        color="0.45",
        linestyle="--",
        label="baseline",
    )

    for index, (row, component_curve) in enumerate(
        zip(
            components.to_dict(orient="records"),
            component_curves,
            strict=True,
        )
    ):
        component_number = int(row["component"])
        center = float(row["center_cm-1"])
        fwhm = float(row["fwhm_cm-1"])

        fit_axis.plot(
            x_plot,
            component_curve,
            color=f"C{index}",
            linestyle=":",
            linewidth=1.8,
            label=(
                f"component {component_number}: "
                f"center={center:.2f}, FWHM={fwhm:.2f}"
            ),
        )
        fit_axis.axvline(
            center,
            color=f"C{index}",
            linestyle=":",
            alpha=0.35,
        )

    fit_axis.set_ylabel("Intensity")
    fit_axis.set_title(title)
    fit_axis.grid(alpha=0.25)
    fit_axis.legend(fontsize=9)
    fit_axis.text(
        0.02,
        0.97,
        "\n".join([
            "X: AsLS baseline + area normalization; no smoothing",
            (
                "selected components = "
                f"{diagnostics['n_deconvolved_components']}"
            ),
            (
                "resolution status = "
                f"{diagnostics['resolution_status']}"
            ),
            (
                "BIC improvement = "
                f"{diagnostics['bic_improvement']:.3g}"
            ),
            f"RMSE = {diagnostics['rmse']:.3g}",
        ]),
        transform=fit_axis.transAxes,
        va="top",
        bbox={
            "facecolor": "white",
            "alpha": 0.85,
            "edgecolor": "0.8",
        },
    )

    residual_axis.axhline(
        0.0,
        color="0.35",
        linewidth=1,
    )
    residual_axis.scatter(
        x_region,
        residual,
        color="tab:purple",
        s=18,
    )
    residual_axis.plot(
        x_region,
        residual,
        color="tab:purple",
        linewidth=1,
    )
    residual_axis.set_xlabel(
        "Raman shift (cm$^{-1}$)"
    )
    residual_axis.set_ylabel("Residual")
    residual_axis.grid(alpha=0.25)

    fig.tight_layout()
    return fig, axes


# 선택한 region에서 가장 강한 subject를 골라 plot한다.
def _candidate_strength(subject_index: int) -> float:
    centers, heights = subject_detections[subject_index]
    mask = (
        (centers >= ANALYZED_RAW_CENTER_CM1 - DECONVOLUTION_HALF_WINDOW_CM1)
        & (centers <= ANALYZED_RAW_CENTER_CM1 + DECONVOLUTION_HALF_WINDOW_CM1)
    )
    if np.any(mask):
        return float(np.max(heights[mask]))
    region_mask = (
        (target_grid >= ANALYZED_RAW_CENTER_CM1 - DECONVOLUTION_HALF_WINDOW_CM1)
        & (target_grid <= ANALYZED_RAW_CENTER_CM1 + DECONVOLUTION_HALF_WINDOW_CM1)
    )
    return float(np.max(X[subject_index, region_mask]))


inspection_subject_index = max(
    raw_diagnostics,
    key=_candidate_strength,
)
inspection_components = raw_component_tables[
    inspection_subject_index
]
inspection_diagnostics = raw_diagnostics[
    inspection_subject_index
]

print({
    "raw_source": (
        "X: AsLS baseline-corrected + area-normalized subject mean (unsmoothed)"
    ),
    "requested_center_cm-1": REQUESTED_RAW_CENTER_CM1,
    "analyzed_center_cm-1": ANALYZED_RAW_CENTER_CM1,
    "inspection_subject_index": inspection_subject_index,
    "candidate_subject_count": int(
        raw_region_summary.loc[
            0,
            "candidate_subject_count",
        ]
    ),
    "fitted_subject_count": len(
        raw_deconvolution_results
    ),
})

display(raw_region_summary)
display(raw_deconvolution_results.head(20))
display(inspection_components)
display(inspection_diagnostics["model_metrics"])

voigt_figure, voigt_axes = plot_voigt_decomposition(
    components=inspection_components,
    diagnostics=inspection_diagnostics,
    title=(
        f"Preprocessed X subject {inspection_subject_index}: "
        f"{ANALYZED_RAW_CENTER_CM1:.1f} cm$^{{-1}}$ region"
    ),
)
voigt_figure.savefig(
    NOTEBOOK_OUTPUT_DIR / "aecd_raw_voigt_decomposition.png",
    dpi=300,
    facecolor="white",
    bbox_inches="tight",
)
plt.show()
plt.close(voigt_figure)


{'raw_source': 'X: AsLS baseline-corrected + area-normalized subject mean (unsmoothed)', 'requested_center_cm-1': 1001.5, 'analyzed_center_cm-1': 1001.5, 'inspection_subject_index': 75, 'candidate_subject_count': 113, 'fitted_subject_count': 113}


,analyzed_center_cm-1,candidate_subject_count,fitted_subject_count,one_component_subjects,two_component_subjects,potentially_resolved_fraction,median_bic_improvement,median_rmse,median_rmse_over_noise,median_center_gap_cm-1
0,1001.5,113,113,77,36,0.0,0.0,0.000016,0.126791,10.309422


,registry_center_cm-1,subject_index,candidate_subject_count,n_deconvolved_components,n_instrument_resolved_peaks,resolution_status,bic_improvement,rmse,rmse_over_noise,minimum_center_gap_cm-1,mean_fitted_fwhm_cm-1,resolution_gap_threshold_cm-1,residual_peak_count
0,1001.5,0,113,1,1,single_component,0.000000,0.000014,0.070294,NaN,25.156140,25.156140,2
1,1001.5,1,113,1,1,single_component,0.000000,0.000010,0.045866,NaN,23.661770,23.661770,0
2,1001.5,2,113,2,1,unresolved,14.396611,0.000007,0.208053,12.239411,33.695602,33.695602,0
3,1001.5,3,113,1,1,single_component,0.000000,0.000008,0.107565,NaN,25.176801,25.176801,0
4,1001.5,4,113,1,1,single_component,0.000000,0.000009,0.043276,NaN,30.148080,30.148080,1
5,1001.5,5,113,2,1,unresolved,33.721578,0.000013,0.108813,11.259760,11.806536,11.806536,0
6,1001.5,6,113,2,1,unresolved,15.700335,0.000009,0.059323,3.055413,17.292049,17.292049,0
7,1001.5,7,113,2,1,unresolved,14.931034,0.000017,0.089987,10.672455,20.206549,20.206549,1
8,1001.5,8,113,1,1,single_component,0.000000,0.000016,0.126791,NaN,26.344786,26.344786,2
9,1001.5,9,113,2,1,unresolved,23.984723,0.000007,0.202939,19.795779,25.687730,25.687730,2


,component,center_cm-1,center_se_cm-1,area,area_se,sigma_cm-1,sigma_se_cm-1,gamma_cm-1,gamma_se_cm-1,fwhm_cm-1
0,1,990.029506,50.779958,0.176752,4.852650,17.419082,189.173577,0.487302,651.033314,41.542334
1,2,1001.950677,0.550228,0.059428,0.441445,0.482833,87.239680,9.682443,36.451369,19.436382


,components,rss,rmse,aic,aicc,bic,delta_bic_from_previous
0,1,7.100826e-08,0.000060,-377.124039,-370.662501,-371.149646,NaN
1,2,1.224259e-08,0.000025,-404.281546,-379.837102,-394.324223,23.174578


C:\Users\user\AppData\Local\Temp\ipykernel_35800\671279095.py:447: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
from pathlib import Path

FIGURE_OUTPUT = NOTEBOOK_OUTPUT_DIR / "aecd_patient_representative_spectrum.png"
PATIENT_REGION_CENTER_CM1 = 1001.5
QC_MAD_THRESHOLD = 3.5

patient_key = subject_keys[inspection_subject_index]
patient_mask = metadata["subject_key"].to_numpy() == patient_key
patient_raw = np.asarray(aligned[patient_mask], dtype=np.float64)
patient_metadata = metadata.loc[patient_mask].reset_index(drop=True)

if patient_raw.ndim != 2 or len(patient_raw) < 2:
    raise RuntimeError("Selected subject needs at least two replicate spectra")

patient_median = np.median(patient_raw, axis=0)
patient_residual = patient_raw - patient_median
patient_scale = np.median(
    np.abs(patient_residual),
    axis=0,
)
patient_scale = np.maximum(patient_scale, np.finfo(float).eps)
patient_distance = np.sqrt(
    np.mean((patient_residual / patient_scale) ** 2, axis=1)
)
distance_median = float(np.median(patient_distance))
distance_mad = float(
    np.median(np.abs(patient_distance - distance_median))
)
if distance_mad == 0.0:
    qc_limit = distance_median + 3.0 * float(np.std(patient_distance))
else:
    qc_limit = distance_median + QC_MAD_THRESHOLD * 1.4826 * distance_mad
qc_pass = patient_distance <= max(qc_limit, distance_median)
if int(qc_pass.sum()) < 2:
    qc_pass[np.argsort(patient_distance)[:2]] = True

qc_spectra = patient_raw[qc_pass]
representative_raw_spectrum = qc_spectra.mean(axis=0)
representative_spectrum = _baseline_area_normalize(
    representative_raw_spectrum,
    grid=source_grid,
)
representative_spectrum = np.interp(
    target_grid,
    source_grid,
    representative_spectrum,
)
qc_pass_count = int(qc_pass.sum())
qc_outlier_count = int((~qc_pass).sum())

representative_components, representative_diagnostics = decompose_peak_region(
    x=target_grid,
    y_values=representative_spectrum,
    left_cm1=PATIENT_REGION_CENTER_CM1 - DECONVOLUTION_HALF_WINDOW_CM1,
    right_cm1=PATIENT_REGION_CENTER_CM1 + DECONVOLUTION_HALF_WINDOW_CM1,
)

region_x = np.asarray(representative_diagnostics["x_region"], dtype=np.float64)
region_y = np.asarray(representative_diagnostics["y_region"], dtype=np.float64)
region_fit = np.asarray(representative_diagnostics["predicted"], dtype=np.float64)
region_residual = np.asarray(representative_diagnostics["residual"], dtype=np.float64)
component_curves = []
component_sum = np.zeros_like(region_x)
plot_x = np.linspace(region_x[0], region_x[-1], 1000)
for component in representative_components.to_dict(orient="records"):
    curve = (
        float(component["area"])
        * voigt_profile(
            plot_x - float(component["center_cm-1"]),
            float(component["sigma_cm-1"]),
            float(component["gamma_cm-1"]),
        )
    )
    component_curves.append(curve)
    component_sum += (
        float(component["area"])
        * voigt_profile(
            region_x - float(component["center_cm-1"]),
            float(component["sigma_cm-1"]),
            float(component["gamma_cm-1"]),
        )
    )
baseline_coefficients = np.polyfit(
    region_x - np.mean(region_x),
    region_fit - component_sum,
    1,
)
baseline = (
    baseline_coefficients[1]
    + baseline_coefficients[0] * (plot_x - np.mean(region_x))
)
total_fit = baseline + np.sum(np.vstack(component_curves), axis=0)

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "svg.fonttype": "none",
})
figure = plt.figure(figsize=(15, 11), facecolor="white")
grid_spec = figure.add_gridspec(
    2,
    2,
    hspace=0.30,
    wspace=0.24,
    left=0.07,
    right=0.98,
    top=0.94,
    bottom=0.13,
)
raw_axis = figure.add_subplot(grid_spec[0, 0])
qc_axis = figure.add_subplot(grid_spec[0, 1])
representative_axis = figure.add_subplot(grid_spec[1, 0])
deconvolution_grid = grid_spec[1, 1].subgridspec(
    2,
    1,
    height_ratios=(3, 1),
    hspace=0.05,
)
deconvolution_axis = figure.add_subplot(deconvolution_grid[0])
residual_axis = figure.add_subplot(
    deconvolution_grid[1],
    sharex=deconvolution_axis,
)

for spectrum in patient_raw:
    raw_axis.plot(source_grid, spectrum, color="0.65", alpha=0.08, linewidth=0.6)
raw_axis.plot(
    source_grid,
    patient_raw.mean(axis=0),
    color="black",
    linewidth=1.8,
    label=f"raw mean (n={len(patient_raw)})",
)
raw_axis.set_title("A. Raw aligned measurements (no smoothing)", loc="left", fontweight="bold")
raw_axis.set_ylabel("Intensity")
raw_axis.legend(frameon=False)

for spectrum, passed in zip(patient_raw, qc_pass, strict=True):
    qc_axis.plot(
        source_grid,
        spectrum,
        color="#9ecae1" if passed else "#d9d9d9",
        alpha=0.55 if passed else 0.35,
        linewidth=0.7,
    )
qc_axis.plot(
    source_grid,
    representative_raw_spectrum,
    color="#08519c",
    linewidth=1.8,
    label=f"QC-passed raw mean (n={qc_pass_count})",
)
qc_axis.set_title("B. QC-passed raw measurements (no smoothing)", loc="left", fontweight="bold")
qc_axis.set_ylabel("Intensity")
qc_axis.legend(frameon=False)
qc_axis.text(
    0.98,
    0.96,
    f"{len(patient_raw)} measurements\nQC filtering\n{qc_pass_count} passed | {qc_outlier_count} outlier(s)",
    transform=qc_axis.transAxes,
    ha="right",
    va="top",
    bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.9},
)

representative_axis.plot(
    target_grid,
    representative_spectrum,
    color="black",
    linewidth=1.5,
)
representative_axis.axvspan(
    PATIENT_REGION_CENTER_CM1 - DECONVOLUTION_HALF_WINDOW_CM1,
    PATIENT_REGION_CENTER_CM1 + DECONVOLUTION_HALF_WINDOW_CM1,
    color="#d73027",
    alpha=0.14,
    label=f"deconvolution region ({PATIENT_REGION_CENTER_CM1:.1f} cm$^{{-1}}$)",
)
representative_axis.set_title("C. Baseline-corrected + area-normalized representative", loc="left", fontweight="bold")
representative_axis.set_xlabel("Raman shift (cm$^{-1}$)")
representative_axis.set_ylabel("Area-normalized intensity")
representative_axis.legend(frameon=False)
representative_axis.text(
    0.02,
    0.96,
    f"AsLS + area normalization from {qc_pass_count} QC-passed measurements",
    transform=representative_axis.transAxes,
    va="top",
)

deconvolution_axis.scatter(
    region_x,
    region_y,
    color="black",
    s=16,
    label="normalized representative (no smoothing)",
    zorder=3,
)
deconvolution_axis.plot(
    plot_x,
    total_fit,
    color="#d73027",
    linewidth=2.0,
    label="total Voigt fit",
)
deconvolution_axis.plot(
    plot_x,
    baseline,
    color="0.45",
    linestyle="--",
    linewidth=1.2,
    label="baseline",
)
for component_index, (component, curve) in enumerate(
    zip(
        representative_components.to_dict(orient="records"),
        component_curves,
        strict=True,
    ),
    start=1,
):
    center = float(component["center_cm-1"])
    deconvolution_axis.plot(
        plot_x,
        curve,
        color=f"C{component_index - 1}",
        linewidth=1.2,
        linestyle=":",
        label=f"component {component_index}: {center:.2f} cm$^{{-1}}$, FWHM {float(component['fwhm_cm-1']):.2f}",
    )
deconvolution_axis.set_title("D. Voigt fit after baseline + area normalization", loc="left", fontweight="bold")
deconvolution_axis.set_ylabel("Area-normalized intensity")
deconvolution_axis.legend(frameon=False, loc="best")
deconvolution_axis.text(
    0.02,
    0.96,
    "\n".join([
        f"BIC improvement = {float(representative_diagnostics['bic_improvement']):.3g}",
        f"RMSE = {float(representative_diagnostics['rmse']):.3g}",
        f"Resolution = {representative_diagnostics['resolution_status']}",
    ]),
    transform=deconvolution_axis.transAxes,
    va="top",
    bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.9},
)
residual_axis.axhline(0.0, color="0.35", linewidth=0.8)
residual_axis.plot(region_x, region_residual, color="#636363", linewidth=0.9)
residual_axis.scatter(region_x, region_residual, color="#636363", s=9)
residual_axis.set_xlabel("Raman shift (cm$^{-1}$)")
residual_axis.set_ylabel("Residual")
residual_axis.grid(alpha=0.2)
plt.setp(deconvolution_axis.get_xticklabels(), visible=False)

for axis in figure.axes:
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.grid(alpha=0.18)

figure.text(
    0.07,
    0.055,
    "Raw and QC-passed panels show aligned measurements without Savitzky-Golay or convolution smoothing. "
    "The dark QC trace is a replicate mean; panel C applies AsLS baseline correction and area normalization. "
    "Panel D shows Voigt model fits only, with resolution status reported from the working FWHM reference.",
    ha="left",
    va="bottom",
    fontsize=9,
    wrap=True,
)
figure.savefig(FIGURE_OUTPUT, dpi=300, facecolor="white", bbox_inches="tight")
plt.show()
print({
    "figure": str(FIGURE_OUTPUT),
    "patient_key": patient_key,
    "raw_measurements": len(patient_raw),
    "qc_passed": qc_pass_count,
    "qc_outliers": qc_outlier_count,
    "deconvolved_components": int(representative_diagnostics["n_deconvolved_components"]),
})

{'figure': 'c:\\Users\\user\\AppData\\Local\\Programs\\Microsoft VS Code\\aecd_api_model_baseline_outputs\\aecd_patient_representative_spectrum.png', 'patient_key': 'subject:76', 'raw_measurements': 121, 'qc_passed': 117, 'qc_outliers': 4, 'deconvolved_components': 2}


C:\Users\user\AppData\Local\Temp\ipykernel_35800\3918145958.py:268: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5. Subject-level clinical evaluation and resolution-aware peak policy

Primary discrimination metrics use nested `StratifiedGroupKFold`: all replicates from one subject remain in one subject-level row, and a subject group never crosses train/test. A single 25% holdout can vary substantially with its random split, so the earlier test AUC 0.818 is not directly comparable to the primary nested OOF AUC unless its exact split, preprocessing, and model-selection rule are recovered.

Peak resolution reference: Guo et al., Anal. Chem. 2020, DOI: 10.1021/acs.analchem.0c02696.
`INSTRUMENT_RESOLUTION_FWHM_CM1 = 2.0` is a working reference until a reference-line FWHM is measured. It sets the minimum candidate-peak distance and contributes to the post-fit resolution status; it is not used to merge neighboring target-grid bins. A two-component fit is called potentially resolved only when BIC supports it and the center gap also exceeds both the working instrument FWHM and the fitted linewidth scale. Peak analysis uses AsLS baseline correction plus area normalization without Savitzky-Golay or convolution smoothing.

In [ ]:
"""
Clinical analysis suite for the AECD prostate SERS cohort.

Deliverables
------------
A) SCREENING (2-class):  [Control + Elevated PSA/Bx-]  vs  Prostate Cancer
     - nested subject-CV out-of-fold (OOF) predictions
     - confusion matrix, screening ROC (AUC + bootstrap CI)
     - sensitivity/specificity/PPV/NPV/bal-acc/F1
     - FALSE-POSITIVE breakdown: of subjects called Cancer, how many were
       truly Control vs Elevated PSA/Bx-

B) 3-CLASS:  Control vs Biopsy-Negative(Elevated PSA/Bx-) vs Prostate Cancer
     - nested OOF 3x3 confusion matrix
     - per-class one-vs-rest ROC + macro AUC
     - per-class sensitivity/precision + balanced accuracy

C) DISCRIMINATIVE PEAKS:  Control vs Elevated PSA/Bx- vs Prostate Cancer
     - univariate Kruskal-Wallis -log10(p) spectrum across the 3 classes
     - multivariate stability via L1-logistic selection frequency (nested)
     - top wavenumbers overlaid on class-mean spectra

Feature representation
----------------------
Full 400-2200 cm-1 range IS the signal window, but substrate broad background
still sits under the analyte peaks. So each spectrum is:
    (1) AsLS baseline-corrected   -> removes substrate background
    (2) area-normalized           -> removes intensity/scaling differences
before modeling. This prevents PCA from locking onto substrate/batch drift.

Model = the config that beat overfitting earlier:
    StandardScaler -> PCA(n_components swept) -> LogisticRegression(strong C)
Inner GridSearchCV picks (n_components, C) with a class-missing-robust
macro-OvR AUC scorer. Everything is subject-grouped (no leakage).

INPUTS (notebook namespace)
---------------------------
    aligned, metadata, subject_keys, target_grid        (rebuilds QC matrix)
    TARGET_COLUMN (default "cohort_group"), RANDOM_STATE (default 0)
    QC_MAD_THRESHOLD (default 3.5)
Outputs are written as PNG/CSV to OUTPUT_DIR (default: current folder).
"""

import os
import numpy as np
import pandas as pd
from scipy import sparse, stats
from scipy.sparse.linalg import spsolve

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    balanced_accuracy_score, f1_score,
)

# ------------------------------------------------------------------
# config
# ------------------------------------------------------------------
OUTPUT_DIR = os.environ.get(
    "AECD_OUTDIR",
    str(globals().get("NOTEBOOK_OUTPUT_DIR", os.path.join(os.getcwd(), "aecd_api_model_baseline_outputs"))),
)
os.makedirs(OUTPUT_DIR, exist_ok=True)
RANDOM_STATE = int(globals().get("RANDOM_STATE", 0))
QC_MAD_THRESHOLD = float(globals().get("QC_MAD_THRESHOLD", 3.5))
TARGET_COLUMN = str(globals().get("TARGET_COLUMN", "cohort_group"))

RAW_CONTROL = "control"
RAW_PDC = "prostate disease control"          # Elevated PSA / Biopsy-negative
RAW_CANCER = "prostate"                        # Prostate Cancer

NICE = {
    RAW_CONTROL: "Control",
    RAW_PDC: "Elevated PSA/Bx-",
    RAW_CANCER: "Prostate Cancer",
}

C_GRID = (3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1)
PCA_GRID = (5, 10, 15, 20)


# ==================================================================
# QC matrix (subject-mean of QC-passed replicates) + fine labels
# ==================================================================
def _qc_mean(reps, mad_threshold=QC_MAD_THRESHOLD):
    reps = np.asarray(reps, dtype=np.float64)
    if reps.ndim != 2 or len(reps) < 2:
        return reps.reshape(-1, reps.shape[-1]).mean(0)
    med = np.median(reps, 0)
    resid = reps - med
    scale = np.maximum(np.median(np.abs(resid), 0), np.finfo(float).eps)
    dist = np.sqrt(np.mean((resid / scale) ** 2, 1))
    dmed = float(np.median(dist))
    dmad = float(np.median(np.abs(dist - dmed)))
    limit = (dmed + 3.0 * float(np.std(dist))) if dmad == 0 \
        else (dmed + mad_threshold * 1.4826 * dmad)
    keep = dist <= max(limit, dmed)
    if keep.sum() < 2:
        keep = np.zeros(len(dist), bool)
        keep[np.argsort(dist)[:2]] = True
    return reps[keep].mean(0)


def build_qc_matrix():
    aligned = np.asarray(globals()["aligned"], dtype=np.float64)
    metadata = globals()["metadata"]
    subj_order = list(globals()["subject_keys"])
    subj_col = metadata["subject_key"].to_numpy()
    label_col = metadata[TARGET_COLUMN].astype(str).to_numpy()

    Xr, yr, keys = [], [], []
    for k in subj_order:
        m = subj_col == k
        if not m.any():
            continue
        Xr.append(_qc_mean(aligned[m]))
        yr.append(pd.Series(label_col[m]).mode().iat[0])
        keys.append(k)
    return (np.vstack(Xr).astype(np.float64),
            np.asarray(yr).astype(str),
            np.asarray(keys))


# ==================================================================
# preprocessing: AsLS baseline + area normalization
# ==================================================================
def asls_baseline(y, lam=1e5, p=0.01, niter=10):
    L = len(y)
    D = sparse.diags([1.0, -2.0, 1.0], [0, -1, -2], shape=(L, L - 2), dtype=float, format="csc")
    D = lam * D.dot(D.T)
    w = np.ones(L, dtype=float)
    for _ in range(niter):
        W = sparse.spdiags(w, 0, L, L).tocsc()
        z = spsolve((W + D).tocsc(), w * y)
        w = p * (y > z) + (1 - p) * (y < z)
    return z


def preprocess(X, lam=1e5, p=0.01):
    """Baseline-correct (remove substrate background) then area-normalize."""
    out = np.empty_like(X, dtype=np.float64)
    for i, row in enumerate(X):
        corr = row - asls_baseline(row, lam=lam, p=p)
        corr = np.clip(corr, 0, None)          # analyte peaks are positive
        area = np.trapezoid(corr)
        out[i] = corr / area if area > 0 else corr
    return out


# ==================================================================
# nested OOF engine
# ==================================================================
def _robust_macro_ovr_auc(estimator, X, y_true):
    proba = estimator.predict_proba(X)
    seen = np.asarray(estimator.classes_)
    gc = np.unique(np.concatenate([seen, np.unique(y_true)]))
    full = np.zeros((proba.shape[0], len(gc)))
    for j, c in enumerate(seen):
        full[:, np.where(gc == c)[0][0]] = proba[:, j]
    if len(gc) == 2:
        # label_binarize intentionally returns one column for binary targets.
        y_binary = (y_true == gc[1]).astype(int)
        if len(np.unique(y_binary)) < 2:
            return np.nan
        return float(roc_auc_score(y_binary, full[:, 1]))
    # Multiclass one-vs-rest branch.
    Y = label_binarize(y_true, classes=gc)
    aucs = [roc_auc_score(Y[:, k], full[:, k])
            for k in range(len(gc)) if len(np.unique(Y[:, k])) > 1]
    return float(np.mean(aucs)) if aucs else np.nan


def _pca_logistic():
    return make_pipeline(
        VarianceThreshold(1e-12),
        StandardScaler(),
        PCA(random_state=RANDOM_STATE),
        LogisticRegression(class_weight="balanced", max_iter=5000,
                           random_state=RANDOM_STATE),
    )


def nested_oof(X, y_enc, groups, n_classes, outer_folds=5, inner_folds=3):
    """Return OOF proba (n x n_classes), per-fold AUC list, chosen params."""
    per_class_min = int(np.bincount(y_enc).min())
    n_out = min(outer_folds, per_class_min)
    outer = StratifiedGroupKFold(n_splits=n_out, shuffle=True,
                                 random_state=RANDOM_STATE)

    oof = np.full((len(X), n_classes), np.nan)
    fold_aucs, chosen = [], []
    for tr, te in outer.split(X, y_enc, groups=groups):
        Xtr, Xte, ytr, yte = X[tr], X[te], y_enc[tr], y_enc[te]
        pca_eff = tuple(pp for pp in PCA_GRID if pp <= len(tr) - 1) or (5,)
        inner = StratifiedGroupKFold(
            n_splits=min(inner_folds, int(np.bincount(ytr).min())),
            shuffle=True, random_state=RANDOM_STATE)
        search = GridSearchCV(
            _pca_logistic(),
            {"pca__n_components": pca_eff,
             "logisticregression__C": C_GRID},
            scoring=_robust_macro_ovr_auc,
            cv=list(inner.split(Xtr, ytr, groups=groups[tr])),
            n_jobs=-1, refit=True, error_score=np.nan,
        )
        search.fit(Xtr, ytr)
        best, seen = search.best_estimator_, np.asarray(
            search.best_estimator_.classes_)
        proba = best.predict_proba(Xte)
        # expand to full class width
        full = np.zeros((len(te), n_classes))
        for j, c in enumerate(seen):
            full[:, c] = proba[:, j]
        oof[te] = full
        fold_aucs.append(_robust_macro_ovr_auc(best, Xte, yte))
        chosen.append(search.best_params_)
    return oof, fold_aucs, chosen


def _boot_auc_ci(y_true_bin, score, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(score)
    stats_ = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true_bin[idx])) < 2:
            continue
        stats_.append(roc_auc_score(y_true_bin[idx], score[idx]))
    lo, hi = np.percentile(stats_, [2.5, 97.5])
    return float(lo), float(hi)


# ==================================================================
# plotting helpers
# ==================================================================
def plot_confusion(cm, labels, title, path, normalize=True):
    fig, ax = plt.subplots(figsize=(1.6 + 1.1 * len(labels),
                                    1.4 + 1.0 * len(labels)))
    disp = cm.astype(float)
    if normalize:
        disp = disp / disp.sum(1, keepdims=True).clip(min=1)
    im = ax.imshow(disp, cmap="Blues", vmin=0, vmax=1 if normalize else None)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha="right"); ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cm[i, j]}\n({disp[i, j]*100:.0f}%)",
                    ha="center", va="center",
                    color="white" if disp[i, j] > 0.5 else "black", fontsize=9)
    fig.colorbar(im, fraction=0.046, pad=0.04)
    fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)


def plot_roc(curves, title, path):
    fig, ax = plt.subplots(figsize=(5.5, 5))
    for name, (fpr, tpr, auc, ci) in curves.items():
        lbl = f"{name}: AUC={auc:.3f}"
        if ci: lbl += f" [{ci[0]:.2f},{ci[1]:.2f}]"
        ax.plot(fpr, tpr, lw=2, label=lbl)
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("1 - Specificity"); ax.set_ylabel("Sensitivity")
    ax.set_title(title); ax.legend(fontsize=8, loc="lower right")
    ax.grid(alpha=0.3); fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)


# ==================================================================
# ANALYSIS A : screening (non-cancer vs cancer)
# ==================================================================
def analysis_screening(Xp, y_fine, groups):
    # binary label: 1 = cancer, 0 = non-cancer (control + PDC)
    y_bin = (y_fine == RAW_CANCER).astype(int)
    oof, fold_aucs, chosen = nested_oof(Xp, y_bin, groups, n_classes=2)
    score = oof[:, 1]                                  # P(cancer)
    valid = ~np.isnan(score)
    yb, sc, yf = y_bin[valid], score[valid], y_fine[valid]

    # ROC + Youden threshold
    fpr, tpr, thr = roc_curve(yb, sc)
    auc = roc_auc_score(yb, sc)
    ci = _boot_auc_ci(yb, sc, seed=RANDOM_STATE)
    j = np.argmax(tpr - fpr)
    cut = thr[j]
    pred = (sc >= cut).astype(int)

    cm = confusion_matrix(yb, pred, labels=[0, 1])     # rows true, cols pred
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    ppv = tp / (tp + fp) if tp + fp else np.nan
    npv = tn / (tn + fn) if tn + fn else np.nan

    # FP breakdown: predicted cancer but truly non-cancer
    called_cancer = pred == 1
    fp_mask = called_cancer & (yb == 0)
    fp_control = int(np.sum(fp_mask & (yf == RAW_CONTROL)))
    fp_pdc = int(np.sum(fp_mask & (yf == RAW_PDC)))
    # also: composition of ALL predicted-cancer
    pc_true_cancer = int(np.sum(called_cancer & (yb == 1)))

    metrics = {
        "n_subjects": int(valid.sum()),
        "n_cancer": int(np.sum(yb == 1)),
        "n_noncancer": int(np.sum(yb == 0)),
        "screening_AUC": auc, "AUC_CI_low": ci[0], "AUC_CI_high": ci[1],
        "per_fold_AUC_mean": float(np.nanmean(fold_aucs)),
        "threshold_youden": float(cut),
        "sensitivity": sens, "specificity": spec, "PPV": ppv, "NPV": npv,
        "balanced_accuracy": balanced_accuracy_score(yb, pred),
        "f1_cancer": f1_score(yb, pred, zero_division=0),
    }

    # plots
    plot_confusion(cm, ["Non-cancer", "Cancer"],
                   "A. Screening confusion (OOF)",
                   os.path.join(OUTPUT_DIR, "A_screening_confusion.png"))
    plot_roc({"Screening (Cancer vs rest)": (fpr, tpr, auc, ci)},
             "A. Screening ROC (nested OOF)",
             os.path.join(OUTPUT_DIR, "A_screening_roc.png"))

    print("\n" + "=" * 62)
    print("A) SCREENING  [Control + Elevated PSA/Bx-]  vs  Prostate Cancer")
    print("=" * 62)
    for k, v in metrics.items():
        print(f"  {k:22s}: {v:.3f}" if isinstance(v, float) else
              f"  {k:22s}: {v}")
    print(f"\n  Predicted CANCER = {int(called_cancer.sum())} subjects")
    print(f"    - truly Prostate Cancer      : {pc_true_cancer}")
    print(f"    - FALSE POS from Control      : {fp_control}")
    print(f"    - FALSE POS from Elevated PSA/Bx- : {fp_pdc}")
    print(f"  (Interpretation: of {int(fp_mask.sum())} false alarms, "
          f"{fp_pdc} were Elevated-PSA/Bx- and {fp_control} were Control)")

    fp_breakdown = {"predicted_cancer_total": int(called_cancer.sum()),
                    "true_cancer": pc_true_cancer,
                    "fp_control": fp_control, "fp_elevated_psa_bx_neg": fp_pdc}
    return metrics, cm, fp_breakdown


# ==================================================================
# ANALYSIS B : 3-class
# ==================================================================
def analysis_three_class(Xp, y_fine, groups):
    order = [RAW_CONTROL, RAW_PDC, RAW_CANCER]
    le = LabelEncoder().fit(order)
    y_enc = le.transform(y_fine)
    oof, fold_aucs, chosen = nested_oof(Xp, y_enc, groups, n_classes=3)
    valid = ~np.isnan(oof[:, 0])
    yv, ov = y_enc[valid], oof[valid]
    pred = ov.argmax(1)

    cm = confusion_matrix(yv, pred, labels=[0, 1, 2])
    labels_nice = [NICE[c] for c in order]

    # per-class OvR ROC
    Y = label_binarize(yv, classes=[0, 1, 2])
    curves, per_class = {}, {}
    for k, c in enumerate(order):
        if len(np.unique(Y[:, k])) < 2:
            continue
        fpr, tpr, _ = roc_curve(Y[:, k], ov[:, k])
        a = roc_auc_score(Y[:, k], ov[:, k])
        ci = _boot_auc_ci(Y[:, k], ov[:, k], seed=RANDOM_STATE)
        curves[NICE[c]] = (fpr, tpr, a, ci)
        sens = cm[k, k] / cm[k].sum() if cm[k].sum() else np.nan
        prec = cm[k, k] / cm[:, k].sum() if cm[:, k].sum() else np.nan
        per_class[NICE[c]] = {"OvR_AUC": a, "AUC_CI": ci,
                              "sensitivity": sens, "precision": prec}

    macro_auc = float(np.mean([v[2] for v in curves.values()]))
    bal = balanced_accuracy_score(yv, pred)
    macro_f1 = f1_score(yv, pred, average="macro", zero_division=0)

    plot_confusion(cm, labels_nice, "B. 3-class confusion (OOF)",
                   os.path.join(OUTPUT_DIR, "B_3class_confusion.png"))
    plot_roc(curves, "B. 3-class one-vs-rest ROC (nested OOF)",
             os.path.join(OUTPUT_DIR, "B_3class_roc.png"))

    print("\n" + "=" * 62)
    print("B) 3-CLASS  Control vs Biopsy-Negative vs Prostate Cancer")
    print("=" * 62)
    print(f"  macro OvR AUC = {macro_auc:.3f} | "
          f"balanced acc = {bal:.3f} | macro F1 = {macro_f1:.3f}")
    for name, d in per_class.items():
        print(f"  {name:20s} AUC={d['OvR_AUC']:.3f} "
              f"[{d['AUC_CI'][0]:.2f},{d['AUC_CI'][1]:.2f}] "
              f"sens={d['sensitivity']:.3f} prec={d['precision']:.3f}")

    summary = {"macro_ovr_auc": macro_auc, "balanced_accuracy": bal,
               "macro_f1": macro_f1, "per_class": per_class}
    return summary, cm


# ==================================================================
# ANALYSIS C : discriminative peaks (3-class)
# ==================================================================
def analysis_peaks(Xp, y_fine, grid, groups, top_k=12):
    order = [RAW_CONTROL, RAW_PDC, RAW_CANCER]
    groups_by_class = [Xp[y_fine == c] for c in order]

    # univariate Kruskal-Wallis across 3 classes, per wavenumber
    pvals = np.ones(Xp.shape[1])
    for j in range(Xp.shape[1]):
        try:
            _, pvals[j] = stats.kruskal(*[g[:, j] for g in groups_by_class])
        except ValueError:
            pvals[j] = 1.0
    neglogp = -np.log10(np.clip(pvals, 1e-300, 1))

    # multivariate stability: L1-logistic selection frequency over outer folds
    y_enc = LabelEncoder().fit(order).transform(y_fine)
    sel_freq = np.zeros(Xp.shape[1])
    n_out = min(5, int(np.bincount(y_enc).min()))
    outer = StratifiedGroupKFold(n_splits=n_out, shuffle=True,
                                 random_state=RANDOM_STATE)
    n_folds = 0
    for tr, _ in outer.split(Xp, y_enc, groups=groups):
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(solver="saga", l1_ratio=1.0,
                               C=0.05, class_weight="balanced",
                               max_iter=5000, random_state=RANDOM_STATE),
        )
        # one-vs-rest coefficients aggregated
        from sklearn.multiclass import OneVsRestClassifier
        ovr = OneVsRestClassifier(clf)
        ovr.fit(Xp[tr], y_enc[tr])
        nonzero = np.zeros(Xp.shape[1], bool)
        for est in ovr.estimators_:
            coef = est.named_steps["logisticregression"].coef_.ravel()
            nonzero |= (coef != 0)
        sel_freq += nonzero
        n_folds += 1
    sel_freq /= max(n_folds, 1)

    # top discriminative wavenumbers by -log10(p)
    top_idx = np.argsort(neglogp)[::-1][:top_k]
    top_idx = np.sort(top_idx)
    peak_table = pd.DataFrame({
        "wavenumber_cm-1": grid[top_idx].round(1),
        "neg_log10_p": neglogp[top_idx].round(2),
        "kw_p_value": pvals[top_idx],
        "l1_selection_freq": sel_freq[top_idx].round(2),
        "mean_Control": groups_by_class[0][:, top_idx].mean(0),
        "mean_ElevatedPSA_Bx-": groups_by_class[1][:, top_idx].mean(0),
        "mean_Cancer": groups_by_class[2][:, top_idx].mean(0),
    }).sort_values("neg_log10_p", ascending=False).reset_index(drop=True)
    peak_table.to_csv(os.path.join(OUTPUT_DIR, "C_discriminative_peaks.csv"),
                      index=False, encoding="utf-8-sig")

    # figure: class-mean spectra + significance track + markers
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(11, 7), sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.08})
    colors = {RAW_CONTROL: "#2c7fb8", RAW_PDC: "#f0a202", RAW_CANCER: "#d7191c"}
    for c in order:
        ax1.plot(grid, Xp[y_fine == c].mean(0), lw=1.4,
                 color=colors[c], label=NICE[c])
    for gi in grid[top_idx]:
        ax1.axvline(gi, color="0.7", lw=0.8, ls=":", zorder=0)
    ax1.set_ylabel("Baseline-corrected,\narea-normalized intensity")
    ax1.set_title("C. Discriminative peaks: Control vs Elevated PSA/Bx- vs Prostate Cancer",
                  loc="left", fontweight="bold")
    ax1.legend(frameon=False)
    annotation_indices = []
    for top_index in top_idx:
        if not annotation_indices or grid[top_index] - grid[annotation_indices[-1]] >= 8.0:
            annotation_indices.append(top_index)
    for gi in grid[annotation_indices]:
        ax1.annotate(f"{gi:.0f}", (gi, ax1.get_ylim()[1]),
                     rotation=90, va="top", ha="center", fontsize=7, color="0.3")

    ax2.plot(grid, neglogp, color="0.35", lw=0.9)
    ax2.scatter(grid[top_idx], neglogp[top_idx], color="#d7191c", s=20, zorder=3)
    sig = -np.log10(0.05)
    ax2.axhline(sig, color="green", ls="--", lw=1, label="p = 0.05")
    ax2.set_ylabel("-log10(p)\nKruskal-Wallis")
    ax2.set_xlabel("Raman shift (cm$^{-1}$)")
    ax2.legend(frameon=False, fontsize=8)
    for ax in (ax1, ax2):
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        ax.grid(alpha=0.15)
    fig.savefig(os.path.join(OUTPUT_DIR, "C_discriminative_peaks.png"),
                dpi=200, bbox_inches="tight")
    plt.close(fig)

    print("\n" + "=" * 62)
    print("C) DISCRIMINATIVE PEAKS  (Control vs Elevated PSA/Bx- vs Cancer)")
    print("=" * 62)
    print(peak_table.to_string(index=False))
    return peak_table


# ==================================================================
# run everything
# ==================================================================
def run_clinical_suite(lam=1e5, p=0.01):
    print("Building QC subject-mean matrix ...")
    X_qc, y_fine, keys = build_qc_matrix()
    print(f"  X_qc={X_qc.shape}  classes="
          f"{dict(zip(*np.unique(y_fine, return_counts=True)))}")

    print("Preprocessing (AsLS baseline + area normalization) ...")
    Xp = preprocess(X_qc, lam=lam, p=p)
    grid = np.asarray(
        globals().get("source_grid", globals()["target_grid"]),
        dtype=np.float64,
    )
    groups = keys

    a_metrics, a_cm, a_fp = analysis_screening(Xp, y_fine, groups)
    b_summary, b_cm = analysis_three_class(Xp, y_fine, groups)
    c_peaks = analysis_peaks(Xp, y_fine, grid, groups)

    print("\nSaved figures/CSV to:", os.path.abspath(OUTPUT_DIR))
    print("  A_screening_confusion.png / A_screening_roc.png")
    print("  B_3class_confusion.png / B_3class_roc.png")
    print("  C_discriminative_peaks.png / C_discriminative_peaks.csv")

    return {"screening": (a_metrics, a_cm, a_fp),
            "three_class": (b_summary, b_cm),
            "peaks": c_peaks}


if not globals().get("_CLINICAL_SUITE_ALREADY_RAN", False):
    results = run_clinical_suite()
    _CLINICAL_SUITE_ALREADY_RAN = True


Building QC subject-mean matrix ...
  X_qc=(113, 933)  classes={np.str_('control'): np.int64(21), np.str_('prostate'): np.int64(43), np.str_('prostate disease control'): np.int64(49)}
Preprocessing (AsLS baseline + area normalization) ...

A) SCREENING  [Control + Elevated PSA/Bx-]  vs  Prostate Cancer
  n_subjects            : 113
  n_cancer              : 43
  n_noncancer           : 70
  screening_AUC         : 0.586
  AUC_CI_low            : 0.481
  AUC_CI_high           : 0.694
  per_fold_AUC_mean     : 0.584
  threshold_youden      : 0.402
  sensitivity           : 0.721
  specificity           : 0.500
  PPV                   : 0.470
  NPV                   : 0.745
  balanced_accuracy     : 0.610
  f1_cancer             : 0.569

  Predicted CANCER = 66 subjects
    - truly Prostate Cancer      : 31
    - FALSE POS from Control      : 9
    - FALSE POS from Elevated PSA/Bx- : 26
  (Interpretation: of 35 false alarms, 26 were Elevated-PSA/Bx- and 9 were Control)

B) 3-CLASS  Contro

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1221: User


C) DISCRIMINATIVE PEAKS  (Control vs Elevated PSA/Bx- vs Cancer)
 wavenumber_cm-1  neg_log10_p  kw_p_value  l1_selection_freq  mean_Control  mean_ElevatedPSA_Bx-  mean_Cancer
          2064.8         2.78    0.001678                1.0      0.000521              0.000586     0.000388
          1979.8         2.68    0.002085                1.0      0.000235              0.000166     0.000259
          1396.6         2.58    0.002650                1.0      0.000255              0.000237     0.000109
          2057.1         2.54    0.002860                1.0      0.000332              0.000410     0.000284
          1398.5         2.43    0.003753                1.0      0.000324              0.000284     0.000142
          1744.2         2.42    0.003839                1.0      0.000305              0.000222     0.000377
          2076.4         2.41    0.003853                1.0      0.000964              0.001186     0.000835
          1939.3         2.40    0.004010             

### 6. 환자별 spectrum 수 민감도 분석

각 환자에서 QC를 통과한 replicate 중 `k`개를 무작위로 뽑아 평균한 뒤, 환자 단위 교차검증으로 분별 성능을 비교합니다. raw 측정 수와 QC 통과 수를 구분하며, 권고값은 임상 cutoff가 아닌 탐색적 포화점입니다.

In [ ]:
from runpy import run_path

helper_candidates = (
    NOTEBOOK_DIR / "aecd_spectra_burden.py",
    Path("aecd_spectra_burden.py"),
)
helper_path = next((path for path in helper_candidates if path.exists()), None)
if helper_path is None:
    raise FileNotFoundError("aecd_spectra_burden.py was not found")
helper_namespace = run_path(str(helper_path))
spectra_burden_results = helper_namespace["run_spectra_burden_analysis"](
    aligned=aligned,
    metadata=metadata,
    subject_keys=np.asarray(subject_keys),
    subject_labels=np.asarray(y),
    target_column=TARGET_COLUMN,
    output_dir=NOTEBOOK_OUTPUT_DIR,
    random_state=RANDOM_STATE,
    qc_mad_threshold=QC_MAD_THRESHOLD,
    n_repeats=3,
)
display(spectra_burden_results["summary"])
print({
    "exploratory_recommended_spectra_per_patient": spectra_burden_results["recommended_spectra"],
    "recommendation_rule": spectra_burden_results["recommendation_rule"],
    "output_dir": spectra_burden_results["output_dir"],
})


FileNotFoundError: aecd_spectra_burden.py was not found

## Checks

- `loaded_spectra`와 API의 `total`을 비교해 `MAX_SPECTRA` 제한 여부를 확인합니다.
- subject당 라벨이 하나가 아닌 행은 자동 제외되므로 제외 원인을 별도로 점검해야 합니다.
- `train_peak_registry`는 train subject에서만 생성되며, test subject는 peak definition에 사용하지 않습니다.
- CNN과 stacking은 동일한 `train_indices`와 `test_indices`를 사용합니다.
- `macro_f1`, `balanced_accuracy`, `macro_ovr_auc`를 함께 보고 class imbalance 영향을 확인합니다.
- 실제 실험 전에는 site·lot·측정일 group split 또는 외부 holdout을 추가해야 합니다.

In [ ]:
checks = {
    "finite_features": bool(np.isfinite(X).all()),
    "unique_subjects": len(subject_keys) == len(set(subject_keys)),
    "subject_level_cv_groups": len(subject_keys) == len(set(subject_keys)),
    "spectra_burden_summary": bool(
        len(spectra_burden_results["summary"]) > 0
    ),
    "demo_mode": DEMO_MODE,
}
if not all(value for key, value in checks.items() if key != "demo_mode"):
    raise RuntimeError(f"Notebook checks failed: {checks}")
print(checks)

{'finite_features': True, 'unique_subjects': True, 'subject_level_cv_groups': True, 'spectra_burden_summary': True, 'demo_mode': False}


## Next Steps

1. `TARGET_COLUMN`, site·cohort 필터와 임상 라벨 정의를 확정합니다.
2. 장비·site·lot 기준 group split 또는 외부 holdout을 추가합니다.
3. 전처리와 모델 하이퍼파라미터를 Pipeline 안에 넣고 교차검증합니다.
4. 실험 결과와 데이터 버전을 MLflow에 기록합니다.

데모 실행 결과는 코드 경로 검증용이며 실제 모델 성능으로 해석하지 않습니다.